# Unit 3. Multiple Regression II: Inference and Interpretation

精算統計模型 (SOA Exam SRM/PA). The text of [unit03.pdf](https://github.com/chang-ye-tu/asm/blob/master/note/unit03.pdf) with every R chunk as a code cell.

<a href="https://colab.research.google.com/github/chang-ye-tu/asm/blob/master/colab/unit03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The notebook runs on the R runtime (on Colab: Runtime › Change runtime type › R). Run the setup cell first: it installs the packages and downloads the data this unit uses.

In [ ]:
# Setup: run this cell first. It installs the packages this unit uses and downloads
# its data files into data/, the paths the code below reads.
local({
  if (is.null(getOption("repos")) || identical(unname(getOption("repos")["CRAN"]), "@CRAN@"))
    options(repos = c(CRAN = "https://cloud.r-project.org"))
  os <- tryCatch(readLines("/etc/os-release"), error = function(e) character())
  code <- sub("^VERSION_CODENAME=", "", grep("^VERSION_CODENAME=", os, value = TRUE))
  if (length(code) == 1 && nzchar(code))  # Linux, e.g. Colab: prebuilt binary packages
    options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", code)),
            HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(), paste(getRversion(),
              R.version$platform, R.version$arch, R.version$os)))
})
pkgs <- c("car", "ISLR2", "MASS")
new <- setdiff(pkgs, rownames(installed.packages()))
if (length(new)) install.packages(new)
data_files <- c("data/frees/csv/Chicago.csv", "data/frees/csv/NAICExpense.csv", "data/frees/csv/Refrigerator.csv", "data/frees/csv/TermLife.csv", "data/frees/csv/UNLifeExpectancy.csv", "data/frees/csv/WiscLottery.csv", "data/islr/Advertising.csv")
for (f in data_files) if (!file.exists(f)) {
  dir.create(dirname(f), recursive = TRUE, showWarnings = FALSE)
  url <- if (startsWith(f, "data/islr/"))
    paste0("https://www.statlearning.com/s/", basename(f)) else
    paste0("https://raw.githubusercontent.com/chang-ye-tu/asm/master/note/", f)
  download.file(url, f, quiet = TRUE, mode = "wb")
}
options(width = 62, digits = 4, scipen = 4, repr.plot.width = 6.4, repr.plot.height = 3.0)

In [ ]:
library(ISLR2)

## 1 Inference for One Coefficient

> **Reading.** FREES §3.4.1–3.4.2 · CLM §7. SRM learning outcome 2(e).

**Remark** (The classical assumptions; Unit 2). A1–A5 are Unit 2's classical assumptions on $\mathbf{Y}=\mathbf{X}\boldsymbol{\beta}+\boldsymbol{\varepsilon}$ with $\mathbf{X}$ fixed: linearity (A1), zero-mean errors (A2), $\operatorname{var}(\boldsymbol{\varepsilon})=\sigma^2\mathbf{I}_n$ with $\sigma^2>0$ (A3), $\operatorname{rank}(\mathbf{X})=k+1$, with $n>k+1$ for inference (A4), and normal errors (A5).

**Example 3.1** (Term life insurance, continued; FREES §3.4.1). The regression of `LNFACE` on `EDUCATION`, `NUMHH` and `LNINCOME` for the $275$ households of Unit 2 that bought term insurance. By Unit 2's definition of standard errors, $\mathrm{SE}(b_j)=s\sqrt{[(\mathbf{X}'\mathbf{X})^{-1}]_{jj}}$, the diagonal being read from the matrix printed below.

In [ ]:
term <- subset(read.csv("data/frees/csv/TermLife.csv"), FACE > 0)
term$LNFACE <- log(term$FACE); term$LNINCOME <- log(term$INCOME)
fit_term <- lm(LNFACE ~ EDUCATION + NUMHH + LNINCOME, data = term)
XtXi_term <- summary(fit_term)$cov.unscaled               # (X'X)^{-1}
round(XtXi_term, 6)
s_term <- summary(fit_term)$sigma
round(cbind(b = coef(fit_term), se = s_term * sqrt(diag(XtXi_term)),
            t = coef(fit_term) / (s_term * sqrt(diag(XtXi_term)))), 4)
c(s = s_term, df = df.residual(fit_term), t_crit = qt(0.975, df.residual(fit_term)))

The fit is reported with each standard error in parentheses below its coefficient:

$$
\widehat{\texttt{LNFACE}}=\underset{(0.846)}{2.584}+\underset{(0.039)}{0.206}\,\texttt{EDUCATION}
+\underset{(0.063)}{0.306}\,\texttt{NUMHH}+\underset{(0.078)}{0.494}\,\texttt{LNINCOME} .
$$

$\mathrm{SE}(b_3)=1.525\sqrt{0.002585}=0.078$ and $t(b_3)=6.36$: `LNINCOME` is $6.4$ standard errors above zero, and $H_0\colon\beta_3=0$ is rejected against $\beta_3\neq0$ at $5\%$ since $6.36>t_{271,0.975}=1.969$.

**Proposition 3.1** (Tests and intervals are dual; FREES §3.4.2; CLM Proposition 7.13). Let $s>0$. For every $d\in\mathbb{R}$, the two-sided $t$ test of $H_0\colon\beta_j=d$ at level $\alpha$ (Unit 2) rejects iff $d$ lies outside the $100(1-\alpha)\%$ confidence interval $\hat\beta_j\pm t_{n-k-1,1-\alpha/2}\mathrm{SE}(\hat\beta_j)$. The interval is therefore the set of hypothesised values that the test does not reject, and $H_0\colon\beta_j=0$ is rejected iff the interval excludes $0$.

**Proof.** Write $c=t_{n-k-1,1-\alpha/2}$ and $\mathrm{SE}=\mathrm{SE}(\hat\beta_j)$, positive when $s>0$. The test rejects iff $|\hat\beta_j-d|/\mathrm{SE}>c$, that is iff $\hat\beta_j-d>c\,\mathrm{SE}$ or $\hat\beta_j-d<-c\,\mathrm{SE}$. The first inequality is $d<\hat\beta_j-c\,\mathrm{SE}$ and the second is $d>\hat\beta_j+c\,\mathrm{SE}$, and together they say that $d$ lies outside $[\hat\beta_j-c\,\mathrm{SE},\ \hat\beta_j+c\,\mathrm{SE}]$. Taking $d=0$ gives the last statement. ∎

**Remark** (FREES §3.4.1–3.4.2). When $\hat\beta_j\neq0$, the estimate and its $t$ ratio determine the standard error, $\mathrm{SE}=\hat\beta_j/t(\hat\beta_j)$, and hence the interval. The quantile $t_{n-k-1,0.975}$ is at least the normal quantile $z_{0.975}=1.95996$, written $1.96$, and tends to it as $n-k-1$ grows (the prerequisites' proposition **quantiles of the $t$ law**), which gives FREES's rule of thumb that a variable is important when its $t$ ratio exceeds $2$ in absolute value:

In [ ]:
round(c(qt(0.975, c(10, 30, 100, 271, 1000)), normal = qnorm(0.975)), 4)
print(c(normal = qnorm(0.975)), digits = 6)

## 2 Added Variable Plots and Partial Correlation

> **Reading.** FREES §3.4.3–3.4.4, §6.1.1 · CLM §8. SRM learning outcome 2(c).

**Theorem 3.2** (Frisch–Waugh–Lovell; CLM Theorem 8.1). Partition $\mathbf{X}=[\mathbf{X}_1\ \mathbf{X}_2]$ and $\hat{\boldsymbol{\beta}}=(\hat{\boldsymbol{\beta}}_1',\hat{\boldsymbol{\beta}}_2')'$ accordingly, and let $\mathbf{M}_2=\mathbf{I}-\mathbf{X}_2(\mathbf{X}_2'\mathbf{X}_2)^{-1}\mathbf{X}_2'$ be the residual maker of $\mathbf{X}_2$. Under A4:

(i) $\mathbf{X}_1'\mathbf{M}_2\mathbf{X}_1$ is invertible and $\hat{\boldsymbol{\beta}}_1=(\mathbf{X}_1'\mathbf{M}_2\mathbf{X}_1)^{-1}\mathbf{X}_1'\mathbf{M}_2\mathbf{Y}$;

(ii) $\hat{\boldsymbol{\beta}}_1$ is the least squares coefficient vector of the regression of $\mathbf{M}_2\mathbf{Y}$ on $\mathbf{M}_2\mathbf{X}_1$, and the residual vector of that regression is the residual vector $\mathbf e$ of the full regression.

**Proof.** **Step 1: $\mathbf{X}_2$ and $\mathbf{M}_2\mathbf{X}_1$ have full column rank.** The columns of $\mathbf{X}_2$ are columns of $\mathbf{X}$, which are linearly independent by A4; so $\mathbf{X}_2'\mathbf{X}_2$ is invertible, as in Step 3 of Unit 2's theorem **the least squares estimator**, and $\mathbf{M}_2$ is defined, symmetric and idempotent with $\mathbf{M}_2\mathbf{X}_2=\mathbf{0}$ by Unit 2's theorem **properties of $\mathbf{H}$ and $\mathbf{M}$** applied to the design $\mathbf{X}_2$. If $\mathbf{M}_2\mathbf{X}_1\mathbf{v}=\mathbf{0}$ then $\mathbf{X}_1\mathbf{v}=(\mathbf{I}-\mathbf{M}_2)\mathbf{X}_1\mathbf{v}=\mathbf{X}_2(\mathbf{X}_2'\mathbf{X}_2)^{-1}\mathbf{X}_2'\mathbf{X}_1\mathbf{v}=\mathbf{X}_2\mathbf{w}$ with $\mathbf{w}=(\mathbf{X}_2'\mathbf{X}_2)^{-1}\mathbf{X}_2'\mathbf{X}_1\mathbf{v}$, so $[\mathbf{X}_1\ \mathbf{X}_2](\mathbf{v}',-\mathbf{w}')'=\mathbf{0}$ and A4 forces $\mathbf{v}=\mathbf{0}$. Hence $\mathbf{v}'\mathbf{X}_1'\mathbf{M}_2\mathbf{X}_1\mathbf{v}=\|\mathbf{M}_2\mathbf{X}_1\mathbf{v}\|^2>0$ for $\mathbf{v}\neq\mathbf{0}$, and $\mathbf{X}_1'\mathbf{M}_2\mathbf{X}_1=(\mathbf{M}_2\mathbf{X}_1)'(\mathbf{M}_2\mathbf{X}_1)$ is positive definite, hence invertible.

**Step 2: the partitioned normal equations.** The normal equations $\mathbf{X}'\mathbf{X}\hat{\boldsymbol{\beta}}=\mathbf{X}'\mathbf{Y}$ read

$$
\mathbf{X}_1'\mathbf{X}_1\hat{\boldsymbol{\beta}}_1+\mathbf{X}_1'\mathbf{X}_2\hat{\boldsymbol{\beta}}_2=\mathbf{X}_1'\mathbf{Y},\qquad
\mathbf{X}_2'\mathbf{X}_1\hat{\boldsymbol{\beta}}_1+\mathbf{X}_2'\mathbf{X}_2\hat{\boldsymbol{\beta}}_2=\mathbf{X}_2'\mathbf{Y} .
$$

The second gives $\hat{\boldsymbol{\beta}}_2=(\mathbf{X}_2'\mathbf{X}_2)^{-1}\mathbf{X}_2'(\mathbf{Y}-\mathbf{X}_1\hat{\boldsymbol{\beta}}_1)$. Substituting into the first, with $\mathbf{H}_2=\mathbf{X}_2(\mathbf{X}_2'\mathbf{X}_2)^{-1}\mathbf{X}_2'=\mathbf{I}-\mathbf{M}_2$,

$$
\mathbf{X}_1'\mathbf{X}_1\hat{\boldsymbol{\beta}}_1+\mathbf{X}_1'\mathbf{H}_2(\mathbf{Y}-\mathbf{X}_1\hat{\boldsymbol{\beta}}_1)=\mathbf{X}_1'\mathbf{Y} .
$$

Replacing $\mathbf{H}_2$ by $\mathbf{I}-\mathbf{M}_2$,

$$
\mathbf{X}_1'\mathbf{X}_1\hat{\boldsymbol{\beta}}_1+\mathbf{X}_1'(\mathbf{Y}-\mathbf{X}_1\hat{\boldsymbol{\beta}}_1)-\mathbf{X}_1'\mathbf{M}_2(\mathbf{Y}-\mathbf{X}_1\hat{\boldsymbol{\beta}}_1)=\mathbf{X}_1'\mathbf{Y} ,
$$

and the first two terms sum to $\mathbf{X}_1'\mathbf{Y}$, leaving $\mathbf{X}_1'\mathbf{M}_2\mathbf{X}_1\hat{\boldsymbol{\beta}}_1=\mathbf{X}_1'\mathbf{M}_2\mathbf{Y}$. Step 1 gives (i).

**Step 3: the partialled regression.** Put $\tilde{\mathbf{Y}}=\mathbf{M}_2\mathbf{Y}$ and $\tilde{\mathbf{X}}_1=\mathbf{M}_2\mathbf{X}_1$, which has full column rank by Step 1. By Unit 2's theorem **the least squares estimator** and $\mathbf{M}_2'\mathbf{M}_2=\mathbf{M}_2$, the coefficient vector of the regression of $\tilde{\mathbf{Y}}$ on $\tilde{\mathbf{X}}_1$ is

$$
(\tilde{\mathbf{X}}_1'\tilde{\mathbf{X}}_1)^{-1}\tilde{\mathbf{X}}_1'\tilde{\mathbf{Y}}
=(\mathbf{X}_1'\mathbf{M}_2\mathbf{X}_1)^{-1}\mathbf{X}_1'\mathbf{M}_2\mathbf{Y}=\hat{\boldsymbol{\beta}}_1 .
$$

Its residual vector is $\tilde{\mathbf e}=\mathbf{M}_2\mathbf{Y}-\mathbf{M}_2\mathbf{X}_1\hat{\boldsymbol{\beta}}_1=\mathbf{M}_2(\mathbf{Y}-\mathbf{X}_1\hat{\boldsymbol{\beta}}_1)$. The full regression gives $\mathbf{Y}-\mathbf{X}_1\hat{\boldsymbol{\beta}}_1=\mathbf{X}_2\hat{\boldsymbol{\beta}}_2+\mathbf e$, so $\tilde{\mathbf e}=\mathbf{M}_2\mathbf{X}_2\hat{\boldsymbol{\beta}}_2+\mathbf{M}_2\mathbf e=\mathbf{M}_2\mathbf e$. Finally $\mathbf{X}_2'\mathbf e=\mathbf{0}$ is part of $\mathbf{X}'\mathbf e=\mathbf{0}$ (Unit 2's **residual identities**), so $\mathbf{M}_2\mathbf e=\mathbf e-\mathbf{X}_2(\mathbf{X}_2'\mathbf{X}_2)^{-1}\mathbf{X}_2'\mathbf e=\mathbf e$, which is (ii). ∎

**Remark** (Holding the other variables fixed; on CLM Theorem 8.1). $\hat{\boldsymbol{\beta}}_1$ is computed from the part of $\mathbf{Y}$ and of $\mathbf{X}_1$ that $\mathbf{X}_2$ does not explain. This is the exact content of “holding the other variables fixed”: no observation need have $\mathbf{X}_2$ fixed, and the adjustment is only as good as the variables in $\mathbf{X}_2$.

**Definition 3.3** (Added variable plot and partial correlation; FREES §3.4.3–3.4.4; CLM Definition 8.9). Under A4, for an explanatory variable $x_j$, let $\mathbf{X}_{(j)}$ be $\mathbf{X}$ without the column of $x_j$ and $\mathbf{M}_{(j)}$ its residual maker. The **added variable plot**, or **partial regression plot**, of $y$ against $x_j$ plots $\mathbf e_1=\mathbf{M}_{(j)}\mathbf{Y}$, the residuals of $y$ regressed on the other explanatory variables, against $\mathbf e_2=\mathbf{M}_{(j)}\mathbf{x}_j$, the residuals of $x_j$ regressed on them. The **partial correlation coefficient** $r(y,x_j\mid\text{others})$ is the sample correlation of $\mathbf e_1$ and $\mathbf e_2$.

**Corollary 3.4** (What an added variable plot shows; CLM Corollary 8.10). Suppose A4 and that the other explanatory variables include the intercept. Then

(i) $\mathbf e_1$ and $\mathbf e_2$ have mean zero;

(ii) the least squares line through the plot has intercept $0$ and slope $\hat\beta_j=\mathbf{x}_j'\mathbf{M}_{(j)}\mathbf{Y}/\mathbf{x}_j'\mathbf{M}_{(j)}\mathbf{x}_j$, the coefficient of $x_j$ in the full regression;

(iii) the residuals of that line are the residuals $\mathbf e$ of the full regression, so its residual sum of squares is $\mathrm{SSE}$;

(iv) under A1–A4, $\operatorname{var}(\hat\beta_j)=\sigma^2/(\mathbf{x}_j'\mathbf{M}_{(j)}\mathbf{x}_j)$.

**Proof.** (i) $\mathbf{1}\in\mathcal{C}(\mathbf{X}_{(j)})$, so $\mathbf{M}_{(j)}\mathbf{1}=\mathbf{0}$ and, by symmetry, $\mathbf{1}'\mathbf{M}_{(j)}=\mathbf{0}'$; hence $\mathbf{1}'\mathbf e_1=\mathbf{1}'\mathbf{M}_{(j)}\mathbf{Y}=0$ and $\mathbf{1}'\mathbf e_2=0$. (ii) The $e_{2i}$ are not all equal, since $\mathbf e_2\neq\mathbf{0}$ by A4 and has mean $0$ by (i). By Unit 2's corollary **the fitted line**, the line fitted to the points $(e_{2i},e_{1i})$ has slope $\sum_i(e_{2i}-\bar e_2)(e_{1i}-\bar e_1)/\sum_i(e_{2i}-\bar e_2)^2$ and intercept $\bar e_1-\text{slope}\cdot\bar e_2$. By (i) the intercept is $0$ and the slope is $\mathbf e_2'\mathbf e_1/\mathbf e_2'\mathbf e_2 =\mathbf{x}_j'\mathbf{M}_{(j)}'\mathbf{M}_{(j)}\mathbf{Y}/\mathbf{x}_j'\mathbf{M}_{(j)}'\mathbf{M}_{(j)}\mathbf{x}_j =\mathbf{x}_j'\mathbf{M}_{(j)}\mathbf{Y}/\mathbf{x}_j'\mathbf{M}_{(j)}\mathbf{x}_j$, which is Theorem 3.2(i) with $\mathbf{X}_1=\mathbf{x}_j$ and $\mathbf{X}_2=\mathbf{X}_{(j)}$. (iii) is Theorem 3.2(ii) for the same partition, the fitted intercept being $0$. (iv) $\hat\beta_j=\mathbf{a}'\mathbf{Y}$ with $\mathbf{a}=\mathbf{M}_{(j)}\mathbf{x}_j/(\mathbf{x}_j'\mathbf{M}_{(j)}\mathbf{x}_j)$, so by the prerequisites' proposition **variance under a linear map** and A3, $\operatorname{var}(\hat\beta_j)=\sigma^2\mathbf{a}'\mathbf{a}=\sigma^2\mathbf{x}_j'\mathbf{M}_{(j)}\mathbf{x}_j/(\mathbf{x}_j'\mathbf{M}_{(j)}\mathbf{x}_j)^2 =\sigma^2/(\mathbf{x}_j'\mathbf{M}_{(j)}\mathbf{x}_j)$. ∎

**Proposition 3.5** (Partial correlation from the $t$ ratio; FREES §3.4.4; CLM Proposition 8.11). Suppose A4, that the other explanatory variables include the intercept and $\mathrm{SSE}>0$, and let $\mathrm{SSE}_{(j)}$ be the residual sum of squares of the regression of $y$ on them and $t(\hat\beta_j)$ the $t$ ratio of $x_j$ in the full regression. Then

$$
r(y,x_j\mid\text{others})^2=\frac{\mathrm{SSE}_{(j)}-\mathrm{SSE}}{\mathrm{SSE}_{(j)}},\qquad
r(y,x_j\mid\text{others})=\frac{t(\hat\beta_j)}{\sqrt{t(\hat\beta_j)^2+n-(k+1)}} . \tag{1}
$$

**Proof.** **Step 1: the correlation.** By Corollary 3.4(i) both residual vectors have mean zero, so their correlation is $r=\mathbf e_2'\mathbf e_1/(\|\mathbf e_2\|\,\|\mathbf e_1\|)$, and $\|\mathbf e_1\|^2=\mathbf{Y}'\mathbf{M}_{(j)}\mathbf{Y}=\mathrm{SSE}_{(j)}$.

**Step 2: the reduction in $\mathrm{SSE}$.** By Unit 2's lemma **adding one explanatory variable**, with the other explanatory variables as the base design and $x_j$ added ($\mathbf{x}_j\notin\mathcal{C}(\mathbf{X}_{(j)})$ by A4), $\mathrm{SSE}_{(j)}-\mathrm{SSE}=(\mathbf{x}_j'\mathbf{M}_{(j)}\mathbf{Y})^2/(\mathbf{x}_j'\mathbf{M}_{(j)}\mathbf{x}_j) =(\mathbf e_2'\mathbf e_1)^2/\|\mathbf e_2\|^2$. Dividing by $\|\mathbf e_1\|^2=\mathrm{SSE}_{(j)}$ and comparing with Step 1 gives the first identity.

**Step 3: the $t$ ratio.** By Unit 2's proposition **$t^2=F$ for one explanatory variable**, $t(\hat\beta_j)^2=(\mathrm{SSE}_{(j)}-\mathrm{SSE})/\{\mathrm{SSE}/(n-k-1)\}$. Write $\Delta=\mathrm{SSE}_{(j)}-\mathrm{SSE}$, so $\Delta/\mathrm{SSE}=t(\hat\beta_j)^2/(n-k-1)$. Then

$$
r^2=\frac{\Delta}{\mathrm{SSE}+\Delta}=\frac{\Delta/\mathrm{SSE}}{1+\Delta/\mathrm{SSE}}
=\frac{t(\hat\beta_j)^2/(n-k-1)}{1+t(\hat\beta_j)^2/(n-k-1)}
=\frac{t(\hat\beta_j)^2}{t(\hat\beta_j)^2+n-k-1} .
$$

**Step 4: the sign.** $r$ has the sign of $\mathbf e_2'\mathbf e_1=\mathbf{x}_j'\mathbf{M}_{(j)}\mathbf{Y}$, which by Corollary 3.4(ii) is the sign of $\hat\beta_j$ and hence of $t(\hat\beta_j)$. Taking the square root with that sign gives the second identity. ∎

**Corollary 3.6** (The standard error through the other explanatory variables; FREES §6.1.1, §5.5.2; CLM Corollary 8.12). Suppose the model has an intercept, and let $R_j^2$ be the coefficient of determination of the regression of $x_j$ on the other explanatory variables and $s_{x_j}$ the standard deviation of $x_j$. Under A4, $R_j^2<1$; and under A1–A4

$$
\operatorname{var}(\hat\beta_j)=\frac{\sigma^2}{(n-1)s_{x_j}^2(1-R_j^2)},\qquad
\mathrm{SE}(\hat\beta_j)=\frac{s\sqrt{\mathrm{VIF}_j}}{s_{x_j}\sqrt{n-1}},\qquad
\mathrm{VIF}_j=\frac{1}{1-R_j^2} , \tag{2}
$$

$\mathrm{VIF}_j$ being the **variance inflation factor** of $x_j$.

**Proof.** $\mathbf{x}_j'\mathbf{M}_{(j)}\mathbf{x}_j=\|\mathbf{M}_{(j)}\mathbf{x}_j\|^2$ is the residual sum of squares $\mathrm{SSE}_j$ of the regression of $x_j$ on the other explanatory variables, which include the intercept. By Unit 2's definition of $R^2$ for that regression, $\mathrm{SSE}_j=(1-R_j^2)\mathrm{SST}_j$ with $\mathrm{SST}_j=\sum_i(x_{ij}-\bar x_j)^2=(n-1)s_{x_j}^2$. If $R_j^2=1$ then $\mathrm{SSE}_j=0$, so $\mathbf{x}_j=\mathbf{H}_{(j)}\mathbf{x}_j\in\mathcal{C}(\mathbf{X}_{(j)})$, a linear dependence among the columns of $\mathbf{X}$ that A4 excludes; hence $R_j^2<1$, and A4 also gives $s_{x_j}>0$. Substituting $\mathbf{x}_j'\mathbf{M}_{(j)}\mathbf{x}_j=(n-1)s_{x_j}^2(1-R_j^2)$ into Corollary 3.4(iv) gives $\operatorname{var}(\hat\beta_j)$, and replacing $\sigma$ by $s$ gives $\mathrm{SE}(\hat\beta_j)$. For $k=1$ the other explanatory variables are the intercept alone, whose fitted values are all $\bar x_1$, so $R_1^2=0$ and (2) is Unit 2's variance of the slope. ∎

**Remark** (FREES §6.1.1). Equation (2) names the three sources of a large standard error: a large residual standard deviation $s$; a large $\mathrm{VIF}_j$, meaning that $x_j$ is nearly a linear combination of the other explanatory variables; and a small sample, $\mathrm{SE}(\hat\beta_j)$ being proportional to $(n-1)^{-1/2}$ at fixed $s$, $s_{x_j}$ and $\mathrm{VIF}_j$. At fixed $\hat\beta_j$ the $t$ ratio therefore grows like $(n-1)^{1/2}$, so in a large sample a variable can be statistically significant with a coefficient too small to matter.

**Example 3.2** (Refrigerator prices; FREES §3.4.3–3.4.4). `PRICE` in dollars of $37$ refrigerators (Consumer Reports, 1992), with the annual energy cost `ECOST` in dollars, the refrigerator and freezer sizes `RSIZE` and `FSIZE` in cubic feet, the number of door shelves `SHELVES` and the number of `FEATURES`.

In [ ]:
fridge <- read.csv("data/frees/csv/Refrigerator.csv")
vars_f <- c("ECOST", "RSIZE", "FSIZE", "SHELVES", "FEATURES", "PRICE")
round(t(sapply(fridge[, vars_f], function(v)
  c(mean = mean(v), median = median(v), sd = sd(v), min = min(v), max = max(v)))), 3)
round(cor(fridge[, vars_f]), 3)
fit_fridge <- lm(PRICE ~ ECOST + RSIZE + FSIZE + SHELVES + FEATURES, data = fridge)
round(coef(summary(fit_fridge)), 4)
c(s = summary(fit_fridge)$sigma, R2 = summary(fit_fridge)$r.squared)

The five explanatory variables explain $83.8\%$ of the variation in price and reduce the typical error from $s_y=\$139.79$ to $s=\$60.65$; every $t$ ratio exceeds $2$ in absolute value. The coefficient of `ECOST` is $-6.96$, while the correlation of `PRICE` with `ECOST` is $+0.522$. The data give negative correlations of `RSIZE` with the other five variables, which FREES Table 3.8 prints as positive (and the `ECOST` entry as $0.333$). The added variable plot of `PRICE` against `ECOST`:

In [ ]:
e1 <- residuals(lm(PRICE ~ RSIZE + FSIZE + SHELVES + FEATURES, data = fridge))
e2 <- residuals(lm(ECOST ~ RSIZE + FSIZE + SHELVES + FEATURES, data = fridge))
c(mean_e1 = mean(e1), mean_e2 = mean(e2), slope = unname(coef(lm(e1 ~ e2))[2]),
  b_ECOST = coef(fit_fridge)[["ECOST"]], cor_e1_e2 = cor(e1, e2),
  SSE_plot = sum(residuals(lm(e1 ~ e2))^2), SSE_full = sum(residuals(fit_fridge)^2))
t_f <- coef(summary(fit_fridge))[-1, "t value"]
round(t_f / sqrt(t_f^2 + 37 - 6), 3)                       # all five partial correlations

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.6)
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.4, 1.2, 0.6), mgp = c(1.9, 0.6, 0))
plot(fridge$ECOST, fridge$PRICE, xlab = "ECOST", ylab = "PRICE", pch = 16, cex = 0.6)
plot(e2, e1, xlab = "ECOST, residual on the others", ylab = "PRICE, residual on the others",
     pch = 16, cex = 0.6)
abline(lm(e1 ~ e2))
par(op)

*Figure 3.1.* Price against energy cost, and the added variable plot of price against energy cost controlling for the other four variables, with its least squares line (FREES Figure 3.4).

The line through the added variable plot has slope $-6.958$, the coefficient of `ECOST`, and residual sum of squares equal to the full $\mathrm{SSE}$, as Corollary 3.4 states. The partial correlation is $-0.481$, and Proposition 3.5 gives it from the $t$ ratio alone, $-3.058/\sqrt{3.058^2+31}=-0.481$; one regression gives all five partial correlations, where the plots need ten.

**Remark** (FREES §3.4.3–3.4.4). The simple correlation of price with energy cost is positive because freezer size is correlated with both ($0.855$ and $0.720$): large freezers cost more to buy and to run. With freezer size and the others held fixed, a higher energy cost goes with a lower price. A variable left out of a regression can reverse the sign of an included one. The plot shows what the partial correlation cannot: curvature and single influential points.

## 3 Inference for Several Coefficients

> **Reading.** ISLR §3.2.2 · CLM §7, §10. SRM learning outcome 2(e).

**Theorem 3.7** (Restricted least squares; CLM Theorem 10.3, equations (47), (38)). Let $\mathbf{R}$ be a $q\times(k+1)$ matrix of rank $q$ and $\mathbf{r}\in\mathbb{R}^q$. Under A4, the minimiser of $\|\mathbf{Y}-\mathbf{X}\boldsymbol{\beta}\|^2$ over $\{\boldsymbol{\beta}:\mathbf{R}\boldsymbol{\beta}=\mathbf{r}\}$ is unique and equals

$$
\hat{\boldsymbol{\beta}}_R=\hat{\boldsymbol{\beta}}-(\mathbf{X}'\mathbf{X})^{-1}\mathbf{R}'\bigl[\mathbf{R}(\mathbf{X}'\mathbf{X})^{-1}\mathbf{R}'\bigr]^{-1}(\mathbf{R}\hat{\boldsymbol{\beta}}-\mathbf{r}), \tag{3}
$$

and its residual sum of squares $\mathrm{SSE}_R$ exceeds the unrestricted $\mathrm{SSE}_U=\mathrm{SSE}$ by

$$
\mathrm{SSE}_R-\mathrm{SSE}_U=(\mathbf{R}\hat{\boldsymbol{\beta}}-\mathbf{r})'\bigl[\mathbf{R}(\mathbf{X}'\mathbf{X})^{-1}\mathbf{R}'\bigr]^{-1}(\mathbf{R}\hat{\boldsymbol{\beta}}-\mathbf{r}) . \tag{4}
$$

**Proof.** **Step 1: $\mathbf{R}(\mathbf{X}'\mathbf{X})^{-1}\mathbf{R}'$ is positive definite.** $\mathbf{A}=\mathbf{X}'\mathbf{X}$ is positive definite by A4, and so is $\mathbf{A}^{-1}$: for $\mathbf{v}\neq\mathbf{0}$, $\mathbf{A}^{-1}\mathbf{v}\neq\mathbf{0}$ and $\mathbf{v}'\mathbf{A}^{-1}\mathbf{v}=(\mathbf{A}^{-1}\mathbf{v})'\mathbf{A}(\mathbf{A}^{-1}\mathbf{v})>0$. The $q$ rows of $\mathbf{R}$ are linearly independent, so $\mathbf{R}'\mathbf{v}\neq\mathbf{0}$ for $\mathbf{v}\neq\mathbf{0}$, and $\mathbf{v}'\mathbf{R}\mathbf{A}^{-1}\mathbf{R}'\mathbf{v}=(\mathbf{R}'\mathbf{v})'\mathbf{A}^{-1}(\mathbf{R}'\mathbf{v})>0$. In particular $\mathbf{R}\mathbf{A}^{-1}\mathbf{R}'$ is invertible.

**Step 2: reduce to a quadratic.** For any $\boldsymbol{\beta}$ put $\mathbf{d}=\boldsymbol{\beta}-\hat{\boldsymbol{\beta}}$. As in Step 4 of Unit 2's theorem **the least squares estimator**, $\|\mathbf{Y}-\mathbf{X}\boldsymbol{\beta}\|^2=\mathrm{SSE}_U-2\mathbf{d}'\mathbf{X}'(\mathbf{Y}-\mathbf{X}\hat{\boldsymbol{\beta}})+\mathbf{d}'\mathbf{A}\mathbf{d}=\mathrm{SSE}_U+\mathbf{d}'\mathbf{A}\mathbf{d}$, the middle term vanishing by the normal equations. The constraint $\mathbf{R}\boldsymbol{\beta}=\mathbf{r}$ is $\mathbf{R}\mathbf{d}=\mathbf{c}$ with $\mathbf{c}=\mathbf{r}-\mathbf{R}\hat{\boldsymbol{\beta}}$. So the problem is to minimise $\mathbf{d}'\mathbf{A}\mathbf{d}$ over $\{\mathbf{d}:\mathbf{R}\mathbf{d}=\mathbf{c}\}$.

**Step 3: the minimiser.** Put $\boldsymbol{\mu}=(\mathbf{R}\mathbf{A}^{-1}\mathbf{R}')^{-1}\mathbf{c}$ and $\mathbf{d}^\ast=\mathbf{A}^{-1}\mathbf{R}'\boldsymbol{\mu}$; then $\mathbf{R}\mathbf{d}^\ast=\mathbf{R}\mathbf{A}^{-1}\mathbf{R}'\boldsymbol{\mu}=\mathbf{c}$, so $\mathbf{d}^\ast$ is feasible. Every feasible $\mathbf{d}$ is $\mathbf{d}^\ast+\mathbf{v}$ with $\mathbf{R}\mathbf{v}=\mathbf{0}$, and $\mathbf{d}^{\ast\prime}\mathbf{A}\mathbf{v}=\boldsymbol{\mu}'\mathbf{R}\mathbf{A}^{-1}\mathbf{A}\mathbf{v}=\boldsymbol{\mu}'\mathbf{R}\mathbf{v}=0$, so

$$
(\mathbf{d}^\ast+\mathbf{v})'\mathbf{A}(\mathbf{d}^\ast+\mathbf{v})=\mathbf{d}^{\ast\prime}\mathbf{A}\mathbf{d}^\ast+2\mathbf{d}^{\ast\prime}\mathbf{A}\mathbf{v}+\mathbf{v}'\mathbf{A}\mathbf{v}
=\mathbf{d}^{\ast\prime}\mathbf{A}\mathbf{d}^\ast+\mathbf{v}'\mathbf{A}\mathbf{v}\geqslant\mathbf{d}^{\ast\prime}\mathbf{A}\mathbf{d}^\ast ,
$$

with equality iff $\mathbf{v}=\mathbf{0}$, $\mathbf{A}$ being positive definite. Hence $\hat{\boldsymbol{\beta}}_R=\hat{\boldsymbol{\beta}}+\mathbf{d}^\ast=\hat{\boldsymbol{\beta}}+\mathbf{A}^{-1}\mathbf{R}'(\mathbf{R}\mathbf{A}^{-1}\mathbf{R}')^{-1}(\mathbf{r}-\mathbf{R}\hat{\boldsymbol{\beta}})$, which is (3).

**Step 4: the gap.** $\mathrm{SSE}_R-\mathrm{SSE}_U=\mathbf{d}^{\ast\prime}\mathbf{A}\mathbf{d}^\ast =\boldsymbol{\mu}'\mathbf{R}\mathbf{A}^{-1}\mathbf{A}\mathbf{A}^{-1}\mathbf{R}'\boldsymbol{\mu}=\boldsymbol{\mu}'(\mathbf{R}\mathbf{A}^{-1}\mathbf{R}')\boldsymbol{\mu} =\mathbf{c}'(\mathbf{R}\mathbf{A}^{-1}\mathbf{R}')^{-1}\mathbf{c}$, and $\mathbf{c}=-(\mathbf{R}\hat{\boldsymbol{\beta}}-\mathbf{r})$ enters a quadratic form, so the sign is immaterial. ∎

**Theorem 3.8** (The general linear hypothesis; ISLR §3.2.2, CLM Theorem 7.4, equation (35)). Let $\mathbf{R}$ be $q\times(k+1)$ of rank $q$ and $\mathbf{r}\in\mathbb{R}^q$. Under A1–A5, for $H_0\colon\mathbf{R}\boldsymbol{\beta}=\mathbf{r}$, the statistic

$$
F=\frac{(\mathbf{R}\hat{\boldsymbol{\beta}}-\mathbf{r})'\bigl[\mathbf{R}(\mathbf{X}'\mathbf{X})^{-1}\mathbf{R}'\bigr]^{-1}(\mathbf{R}\hat{\boldsymbol{\beta}}-\mathbf{r})}{q\,s^2}
\sim F_{q,\,n-k-1}\quad\text{under }H_0 ; \tag{5}
$$

it is defined when $s>0$, which holds with probability one, $\mathrm{SSE}/\sigma^2$ being a sum of $n-k-1\geqslant1$ squared independent standard normals.

**Proof.** **Step 1: the numerator vector.** By Unit 2's theorem **normality of $\hat{\boldsymbol{\beta}}$** and the prerequisites' theorem **closure under affine maps**, $\mathbf{R}\hat{\boldsymbol{\beta}}-\mathbf{r}\sim N(\mathbf{R}\boldsymbol{\beta}-\mathbf{r},\ \sigma^2\boldsymbol{\Sigma}_R)$ with $\boldsymbol{\Sigma}_R=\mathbf{R}(\mathbf{X}'\mathbf{X})^{-1}\mathbf{R}'$, and under $H_0$ the mean is $\mathbf{0}$.

**Step 2: a chi-square.** $\boldsymbol{\Sigma}_R$ is positive definite by Step 1 of Theorem 3.7; let $\boldsymbol{\Sigma}_R^{1/2}$ be its symmetric positive definite square root (prerequisites' theorem **symmetric square root**) and $\mathbf{w}=\sigma^{-1}\boldsymbol{\Sigma}_R^{-1/2}(\mathbf{R}\hat{\boldsymbol{\beta}}-\mathbf{r})$. Under $H_0$, by **closure under affine maps**,

$$
\mathbf{w}\sim N\bigl(\mathbf{0},\ \sigma^{-2}\boldsymbol{\Sigma}_R^{-1/2}\,\sigma^2\boldsymbol{\Sigma}_R\,\boldsymbol{\Sigma}_R^{-1/2}\bigr)=N(\mathbf{0},\mathbf{I}_q),
$$

and by the prerequisites' theorem **chi-square quadratic forms** with the identity matrix, $W=\mathbf{w}'\mathbf{w}=(\mathbf{R}\hat{\boldsymbol{\beta}}-\mathbf{r})'\boldsymbol{\Sigma}_R^{-1}(\mathbf{R}\hat{\boldsymbol{\beta}}-\mathbf{r})/\sigma^2\sim\chi^2_q$.

**Step 3: independence and the ratio.** $W$ is a function of $\hat{\boldsymbol{\beta}}$, and by Unit 2's theorem **distribution of $\mathrm{SSE}$ and independence from $\hat{\boldsymbol{\beta}}$**, $V=\mathrm{SSE}/\sigma^2\sim\chi^2_{n-k-1}$ is independent of $\hat{\boldsymbol{\beta}}$, hence of $W$. By the prerequisites' definition of the $F$ law, $(W/q)/\{V/(n-k-1)\}\sim F_{q,n-k-1}$, and

$$
\frac{W/q}{V/(n-k-1)}=\frac{(\mathbf{R}\hat{\boldsymbol{\beta}}-\mathbf{r})'\boldsymbol{\Sigma}_R^{-1}(\mathbf{R}\hat{\boldsymbol{\beta}}-\mathbf{r})}{q\sigma^2}\cdot
\frac{\sigma^2(n-k-1)}{\mathrm{SSE}}=\frac{(\mathbf{R}\hat{\boldsymbol{\beta}}-\mathbf{r})'\boldsymbol{\Sigma}_R^{-1}(\mathbf{R}\hat{\boldsymbol{\beta}}-\mathbf{r})}{q\,s^2} .
$$

∎

**Corollary 3.9** (Extra sum of squares; ISLR §3.2.2; CLM Corollary 7.6, Corollary 7.16). The statistic (5) equals, for every data set,

$$
F=\frac{(\mathrm{SSE}_R-\mathrm{SSE}_U)/q}{\mathrm{SSE}_U/(n-k-1)}=\frac{(R^2_U-R^2_R)/q}{(1-R^2_U)/(n-k-1)} , \tag{6}
$$

with $R^2=1-\mathrm{SSE}/\mathrm{SST}$ for each of the unrestricted and the restricted fit, the second form when $\mathrm{SST}>0$. For $H_0\colon\beta_{j}=0$ for all $j$ in a set $J$ of $q$ slopes, $\mathrm{SSE}_R$ is the residual sum of squares of the regression on the explanatory variables not in $J$.

**Proof.** By (4) the numerator quadratic form of (5) is $\mathrm{SSE}_R-\mathrm{SSE}_U$, and $q\,s^2=q\,\mathrm{SSE}_U/(n-k-1)$, which gives the first form. Both fits have the same response and hence the same $\mathrm{SST}$; dividing numerator and denominator by $\mathrm{SST}$ and using $R^2=1-\mathrm{SSE}/\mathrm{SST}$ gives the second. When the constraints set the coefficients in $J$ to zero, $\{\mathbf{X}\boldsymbol{\beta}:\mathbf{R}\boldsymbol{\beta}=\mathbf{0}\}$ is the column space of $\mathbf{X}$ with the columns in $J$ removed, and minimising over it is the least squares regression on the remaining columns. ∎

**Remark** (Which $\mathbf{R}$ gives which test; ISLR §3.2.2). $\mathbf{R}=[\mathbf{0}\ \mathbf{I}_k]$ with $\mathbf{r}=\mathbf{0}$ is the overall $F$ test of Unit 2: the restricted fit is $\bar Y\mathbf{1}$, so $\mathrm{SSE}_R=\mathrm{SST}$ and (6) is $\mathrm{MSR}/\mathrm{MSE}$. A single slope, $q=1$, gives the square of its $t$ ratio. Dropping $q$ explanatory variables is ISLR's equation (3.24), the **partial** $F$ test. Hypotheses such as $\beta_1=\beta_2$ or $\beta_1=1$ are not exclusions; FREES treats them in its chapter 4, outside the SRM reading list, and Theorem 3.8 covers them with the appropriate $\mathbf{R}$ and $\mathbf{r}$.

**Remark** (Deciding on important variables; ISLR §3.2.2). ISLR's second question about a multiple regression, which predictors matter, is **variable selection**. With $k$ predictors there are $2^k$ models, too many to fit when $k$ is large, and ISLR names three ways through them, each repeating one step until a stopping rule is met: **forward selection**, which starts from the null model (the intercept alone) and at each step adds the variable giving the lowest residual sum of squares; **backward selection**, which starts from all $k$ and at each step removes the variable with the largest $p$-value; and **mixed selection**, which adds as forward selection does but drops any variable whose $p$-value rises above a threshold. Its third and fourth questions, model fit and prediction, are $R^2$, the residual standard error and the intervals of Unit 2. Selection, with criteria for choosing among the models, is Unit 5's subject.

## 4 Special Explanatory Variables

> **Reading.** FREES §3.5 · ISLR §3.3.1–3.3.2. SRM learning outcomes 2(c), 2(e).

### 4.1 Qualitative explanatory variables

**Definition 3.10** (Qualitative predictors and indicator coding; FREES §3.5.1, ISLR §3.3.1; CLM Definition 11.1). A **qualitative** explanatory variable, or **factor**, takes one of $c$ values, its **levels**. The **indicator** or **dummy** variable of level $l$ is $D_{l}=1$ for an observation at level $l$ and $0$ otherwise. **Baseline coding** enters the intercept and the indicators of all levels but one, the **baseline**. A **binary** variable is a factor with $c=2$, entered as one indicator.

**Proposition 3.11** (Coefficients of a factor are differences of means; ISLR §3.3.1; CLM Proposition 11.2). Let $y$ be regressed on the intercept and the indicators $D_2,\ldots,D_c$ of a factor with baseline level $1$, every level being observed, with $n_l$ observations and mean response $\bar y_l$ at level $l$. Then the fitted value of each observation is the mean of its level, $\hat\beta_0=\bar y_1$, $\hat\beta_l=\bar y_l-\bar y_1$ for $l\geqslant2$, and $\mathrm{SSE}=\sum_l\sum_{i\in\text{level }l}(y_i-\bar y_l)^2$.

**Proof.** **Step 1: the column space.** Every observation is at exactly one level, so $D_1+\cdots+D_c=\mathbf{1}$ and $D_1=\mathbf{1}-D_2-\cdots-D_c$. Hence $\mathcal{C}([\mathbf{1}\ D_2\ \cdots\ D_c])=\mathcal{C}([D_1\ \cdots\ D_c])$, and the $c$ columns $D_1,\ldots,D_c$ are linearly independent, having disjoint supports, each non-empty; so A4 holds for the design, which has $c$ columns.

**Step 2: the projection.** The indicators are orthogonal, $D_l'D_m=0$ for $l\neq m$, with $D_l'D_l=n_l$. For $\mathbf{P}\mathbf{Y}=\sum_lD_l\,\bar y_l$, where $\bar y_l=D_l'\mathbf{Y}/n_l$, the vector $\mathbf{Y}-\mathbf{P}\mathbf{Y}$ is orthogonal to each $D_m$: $D_m'(\mathbf{Y}-\mathbf{P}\mathbf{Y})=D_m'\mathbf{Y}-n_m\bar y_m=0$. By Unit 2's theorem **least squares as projection**, $\hat{\mathbf{Y}}=\mathbf{P}\mathbf{Y}$: each fitted value is the mean of its level, and $\mathrm{SSE}$ is the within-level sum of squares.

**Step 3: the coefficients.** At level $1$ all indicators $D_2,\ldots,D_c$ are $0$, so the fitted value is $\hat\beta_0$, which must equal $\bar y_1$; at level $l\geqslant2$ it is $\hat\beta_0+\hat\beta_l=\bar y_l$. These equations have one solution, which by uniqueness of the least squares coefficients under A4 is $\hat{\boldsymbol{\beta}}$. ∎

**Proposition 3.12** (Codings are reparametrisations; ISLR §3.3.1; CLM Proposition 11.4, Corollary 11.5). Let $\tilde{\mathbf{X}}=\mathbf{X}\mathbf{A}$ with $\mathbf{A}$ an invertible $(k+1)\times(k+1)$ matrix, and let $\mathbf{X}$ satisfy A4. Then $\tilde{\mathbf{X}}$ satisfies A4; the two regressions have the same fitted values, residuals, $\mathrm{SSE}$ and $R^2$; and $\hat{\tilde{\boldsymbol{\beta}}}=\mathbf{A}^{-1}\hat{\boldsymbol{\beta}}$. Changing the baseline of a factor, coding a binary variable as $\pm1$ instead of $1/0$, rescaling a column, and centring a column when the design contains the intercept are all of this form, so none of them changes the fit, and the $F$ test that all $c-1$ coefficients of a factor are zero does not depend on the coding; the individual coefficients and $t$ ratios do.

**Proof.** **Step 1: the column space.** $\tilde{\mathbf{X}}\mathbf{v}=\mathbf{X}(\mathbf{A}\mathbf{v})$ shows $\mathcal{C}(\tilde{\mathbf{X}})\subseteq\mathcal{C}(\mathbf{X})$, and $\mathbf{X}\mathbf{v}=\tilde{\mathbf{X}}(\mathbf{A}^{-1}\mathbf{v})$ the reverse inclusion. If $\tilde{\mathbf{X}}\mathbf{v}=\mathbf{0}$ then $\mathbf{X}(\mathbf{A}\mathbf{v})=\mathbf{0}$, so $\mathbf{A}\mathbf{v}=\mathbf{0}$ and $\mathbf{v}=\mathbf{0}$: A4 holds for $\tilde{\mathbf{X}}$.

**Step 2: the fit.** By Unit 2's theorem **least squares as projection**, the fitted vector is the unique point of the common column space closest to $\mathbf{Y}$, so the fitted values, residuals and $\mathrm{SSE}$ agree, and $\mathrm{SST}$ does not involve the design. Also

$$
\hat{\tilde{\boldsymbol{\beta}}}=(\mathbf{A}'\mathbf{X}'\mathbf{X}\mathbf{A})^{-1}\mathbf{A}'\mathbf{X}'\mathbf{Y}=\mathbf{A}^{-1}(\mathbf{X}'\mathbf{X})^{-1}\mathbf{A}'^{-1}\mathbf{A}'\mathbf{X}'\mathbf{Y}
=\mathbf{A}^{-1}\hat{\boldsymbol{\beta}} .
$$

**Step 3: the examples.** For a factor with baseline $1$, replacing $D_2$ by $D_1$ means expressing $D_1=\mathbf{1}-D_2-\cdots-D_c$ in the old columns and $D_2=\mathbf{1}-D_1-D_3-\cdots-D_c$ in the new: each design's columns are combinations of the other's, so $\tilde{\mathbf{X}}=\mathbf{X}\mathbf{A}$ and $\mathbf{X}=\tilde{\mathbf{X}}\mathbf{B}$ for some matrices $\mathbf{A}$, $\mathbf{B}$; then $\mathbf{X}(\mathbf{A}\mathbf{B}-\mathbf{I})=\mathbf{0}$, and A4 gives $\mathbf{A}\mathbf{B}=\mathbf{I}$, so $\mathbf{A}$ is invertible. For a binary indicator $D$, the $\pm1$ variable is $2D-\mathbf{1}$, and $[\mathbf{1}\ \ 2D-\mathbf{1}]=[\mathbf{1}\ \ D]\begin{pmatrix}1&-1\\0&2\end{pmatrix}$. Centring replaces $\mathbf{x}_j$ by $\mathbf{x}_j-\bar x_j\mathbf{1}$ and rescaling by $a\mathbf{x}_j$ with $a\neq0$; when $\mathbf{1}$ is in the design these are column operations undone by adding back $\bar x_j\mathbf{1}$ and by dividing by $a$.

**Step 4: the factor test.** The restricted model for $H_0$ that the factor has no effect contains the other explanatory variables only, the same for every coding, and the unrestricted fits agree by Step 2. So $\mathrm{SSE}_R$ and $\mathrm{SSE}_U$, and with them the $F$ statistic (6), do not depend on the coding. The coefficients $\mathbf{A}^{-1}\hat{\boldsymbol{\beta}}$ do. ∎

**Proposition 3.13** (The dummy variable trap; ISLR §3.3.1, CLM Remark 10.2). A design containing the intercept and the indicators of all $c$ levels of a factor violates A4.

**Proof.** $D_1+\cdots+D_c-\mathbf{1}=\mathbf{0}$ is a vanishing linear combination of columns of $\mathbf{X}$ with non-zero coefficients. ∎

**Example 3.3** (House ownership and region; ISLR §3.3.1). Average credit card `Balance` of $400$ card holders in the `Credit` data, with house ownership `Own` and `Region` (East, South, West).

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
fit_own <- lm(Balance ~ Own, data = Credit)
round(coef(summary(fit_own)), 4)
round(tapply(Credit$Balance, Credit$Own, mean), 3)
table(Credit$Own)
own_pm <- ifelse(Credit$Own == "Yes", 1, -1)
fit_pm <- lm(Balance ~ own_pm, data = Credit)
round(coef(summary(fit_pm)), 4)
c(same_fit = max(abs(fitted(fit_own) - fitted(fit_pm))))
fit_region <- lm(Balance ~ Region, data = Credit)
round(coef(summary(fit_region)), 4)
round(tapply(Credit$Balance, Credit$Region, mean), 3)
anova(lm(Balance ~ 1, data = Credit), fit_region)
fit_region_W <- lm(Balance ~ relevel(Region, ref = "West"), data = Credit)
round(coef(summary(fit_region_W)), 4)

Non-owners carry \$509.80 on average and owners \$19.73 more, with $p=0.67$: no evidence of a difference. With the $\pm1$ coding the intercept is $519.67$, the average of the two group means (the overall mean only when the groups are equal in size or their means coincide; here the sizes are $193$ and $207$), and the slope is $9.87$, half the difference; the fitted values are identical, as Proposition 3.12 requires. For region, the baseline East has mean $531.00$, and the South and West coefficients are $-12.50$ and $-18.69$, the differences of their means from East's by Proposition 3.11. With West as baseline every coefficient and $t$ ratio changes, while the $F$ test that region has no effect is $F=0.04$ on $(2,397)$ degrees of freedom, $p=0.96$, under either coding.

**Example 3.4** (Term life insurance, continued: single respondents; FREES §3.5.1). $\texttt{SINGLE}=1$ for the $57$ respondents who are neither married nor living with a partner. In $\operatorname{\mathsf{E}}[y]=\beta_0+\beta_1\texttt{LNINCOME}+\beta_2\texttt{SINGLE}$, the regression function is $\beta_0+\beta_1\texttt{LNINCOME}$ for others and $(\beta_0+\beta_2)+\beta_1\texttt{LNINCOME}$ for single respondents: two parallel lines, $\beta_2$ being the expected change in $y$ from the base level to the alternative at every income.

In [ ]:
term$SINGLE <- as.numeric(term$MARSTAT == 0)
table(term$SINGLE)
fit_single <- lm(LNFACE ~ LNINCOME + SINGLE, data = term)
round(coef(summary(fit_single)), 4)
c(exp_minus_0.80 = exp(-0.80))

$\widehat{\texttt{LNFACE}}=5.09+0.634\,\texttt{LNINCOME}-0.800\,\texttt{SINGLE}$, with standard errors $0.89$, $0.078$ and $0.248$: at the same income a single respondent's expected logarithmic face amount is $0.80$ lower, a face amount lower by the factor $e^{-0.80}=0.45$.

### 4.2 Transformed explanatory variables

**Definition 3.14** (Linear in the parameters; FREES §3.5.2, ISLR §3.3.2; CLM Definition 11.6). A model $\operatorname{\mathsf{E}}[y]=\beta_0+\beta_1g_1(\mathbf{x})+\cdots+\beta_pg_p(\mathbf{x})$ with known functions $g_1,\ldots,g_p$ is a linear regression model with explanatory variables $g_1(\mathbf{x}),\ldots,g_p(\mathbf{x})$, and all of Units 2 and 3 applies to it. Examples: **polynomial regression**, $g_j(x)=x^j$; the logarithm, $g(x)=\ln x$; an indicator of a threshold, $g(x)=1\{x\geqslant \xi\}$; and the kink $g(x)=1\{x\geqslant \xi\}(x-\xi)$. A model such as $y=\beta_0+\exp(\beta_1x)+\varepsilon$ is not linear in $\beta_1$ and is outside this class.

**Proposition 3.15** (Piecewise linear regression; FREES §3.5.3; CLM Proposition 11.7). With $z=1\{x\geqslant \xi\}$: the regression function $\operatorname{\mathsf{E}}[y]=\beta_0+\beta_1x+\beta_2z(x-\xi)$ is continuous in $x$, with slope $\beta_1$ below $\xi$ and $\beta_1+\beta_2$ above; the regression function $\operatorname{\mathsf{E}}[y]=\beta_0+\beta_1x+\beta_2z+\beta_3zx$ has slopes $\beta_1$ and $\beta_1+\beta_3$ and a jump of $\beta_2+\beta_3\xi$ at $x=\xi$.

**Proof.** For $x<\xi$, $z=0$ and both functions are $\beta_0+\beta_1x$. For $x\geqslant \xi$, $z=1$: the first is $(\beta_0-\beta_2\xi)+(\beta_1+\beta_2)x$, whose value at $\xi$ is $\beta_0+\beta_1\xi$, the limit from the left; the second is $(\beta_0+\beta_2)+(\beta_1+\beta_3)x$, whose value at $\xi$ exceeds the left limit $\beta_0+\beta_1\xi$ by $\beta_2+\beta_3\xi$. ∎

**Remark** (FREES §3.5.2–3.5.3). FREES's examples: charitable contributions against wages with a kink at the $2007$ social security wage base, \$97,500, above which a marginal dollar of wages is free of social security tax; a federal tax schedule with two kinks; and commissions on share trades, with a jump at the $100$-share round lot. Replacing years of education by $1\{x\geqslant13\}$ trades information for reliability when $x$ is self-reported with error.

**Example 3.5** (Polynomial regression; ISLR §3.3.2). `mpg` against `horsepower` for the $392$ cars of `Auto`.

In [ ]:
fit_hp1 <- lm(mpg ~ horsepower, data = Auto)
fit_hp2 <- lm(mpg ~ horsepower + I(horsepower^2), data = Auto)
fit_hp5 <- lm(mpg ~ poly(horsepower, 5), data = Auto)
round(coef(summary(fit_hp2)), 4)
c(R2_linear = summary(fit_hp1)$r.squared, R2_quadratic = summary(fit_hp2)$r.squared,
  R2_degree5 = summary(fit_hp5)$r.squared)
anova(fit_hp2, fit_hp5)
loocv <- function(m) mean((residuals(m) / (1 - hatvalues(m)))^2)
round(sapply(1:5, function(d) loocv(lm(mpg ~ poly(horsepower, d), data = Auto))), 3)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.6)
op <- par(mar = c(3.2, 3.4, 1.2, 0.6), mgp = c(1.9, 0.6, 0))
plot(Auto$horsepower, Auto$mpg, xlab = "horsepower", ylab = "mpg", pch = 16, cex = 0.4,
     col = "grey45")
hp <- seq(min(Auto$horsepower), max(Auto$horsepower), length.out = 300)
nd_hp <- data.frame(horsepower = hp)
lines(hp, predict(fit_hp1, nd_hp), col = "#B45309", lwd = 2)
lines(hp, predict(fit_hp2, nd_hp), col = "steelblue", lwd = 2)
lines(hp, predict(fit_hp5, nd_hp), col = "darkgreen", lwd = 2)
legend("topright", c("linear", "degree 2", "degree 5"), col = c("#B45309", "steelblue",
       "darkgreen"), lwd = 2, bty = "n", cex = 0.7)
par(op)

*Figure 3.2.* Linear, quadratic and fifth-degree polynomial fits of `mpg` on `horsepower` (ISLR Figure 3.8).

The quadratic term has $t=10.1$, and $R^2$ rises from $0.606$ to $0.688$. The fifth-degree fit is wigglier at the ends of the range, which ISLR calls unnecessary; the partial $F$ test of its three extra terms (Corollary 3.9) gives $F=3.9$ on $(3,386)$ degrees of freedom, $p=0.009$, and Unit 1's leave-one-out shortcut gives a cross-validation error of $19.03$ at degree $5$ against $19.25$ at degree $2$, so on this evidence the extra terms do improve the fit slightly.

### 4.3 Interactions

**Definition 3.16** (Interaction; FREES §3.5.3, ISLR §3.3.2; CLM Definition 11.8). An **interaction term** is the product $x_1x_2$ of two explanatory variables entered as a further explanatory variable, as in $\operatorname{\mathsf{E}}[y]=\beta_0+\beta_1x_1+\beta_2x_2+\beta_3x_1x_2$; $x_1$ and $x_2$ are its **main effects**. A model without interactions is **additive**.

**Proposition 3.17** (Effects in an interaction model; FREES §3.5.3, ISLR §3.3.2; CLM Proposition 11.9). In $\operatorname{\mathsf{E}}[y]=\beta_0+\beta_1x_1+\beta_2x_2+\beta_3x_1x_2$, $\partial\operatorname{\mathsf{E}}[y]/\partial x_1=\beta_1+\beta_3x_2$ and $\partial\operatorname{\mathsf{E}}[y]/\partial x_2=\beta_2+\beta_3x_1$. When $x_2=z$ is an indicator, the regression function is $\beta_0+\beta_1x_1$ at $z=0$ and $(\beta_0+\beta_2)+(\beta_1+\beta_3)x_1$ at $z=1$: two lines differing in intercept by $\beta_2$ and in slope by $\beta_3$.

**Proof.** Differentiate term by term: $\partial(\beta_3x_1x_2)/\partial x_1=\beta_3x_2$, and symmetrically in $x_2$. Substituting $z=0$ and $z=1$ gives the two lines. ∎

**Proposition 3.18** (The hierarchical principle; ISLR §3.3.2; CLM Proposition 11.10). Let the design satisfy A4 and contain $\mathbf{1}$, $\mathbf{x}_1$, $\mathbf{x}_2$ and $\mathbf{x}_1\mathbf{x}_2$ (elementwise product), possibly with further columns that do not involve $x_1$ or $x_2$, and replace $x_1$ by $x_1+a$ and $x_2$ by $x_2+b$ for constants $a$, $b$. Then the fitted values, residuals, $\mathrm{SSE}$, $R^2$, and the coefficient, standard error and $t$ ratio of the interaction are unchanged, while the main-effect coefficients become $\hat\beta_1-b\hat\beta_3$ and $\hat\beta_2-a\hat\beta_3$. If the main effects are left out, the column space $\mathcal{C}([\mathbf{1}\ \ (\mathbf{x}_1+a\mathbf{1})(\mathbf{x}_2+b\mathbf{1})])$ depends on $a$ and $b$ unless $b\mathbf{x}_1+a\mathbf{x}_2\in\mathcal{C}([\mathbf{1}\ \ \mathbf{x}_1\mathbf{x}_2])$, and in general so does the fit.

**Proof.** **Step 1: the shifted columns.** Since $(\mathbf{x}_1+a\mathbf{1})(\mathbf{x}_2+b\mathbf{1})=\mathbf{x}_1\mathbf{x}_2+b\mathbf{x}_1+a\mathbf{x}_2+ab\mathbf{1}$, the shifted design $[\mathbf{1}\ \ \mathbf{x}_1+a\mathbf{1}\ \ \mathbf{x}_2+b\mathbf{1}\ \ (\mathbf{x}_1+a\mathbf{1})(\mathbf{x}_2+b\mathbf{1})]$ is $[\mathbf{1}\ \mathbf{x}_1\ \mathbf{x}_2\ \mathbf{x}_1\mathbf{x}_2]\mathbf{A}$ with

$$
\mathbf{A}=\begin{pmatrix}1&a&b&ab\\0&1&0&b\\0&0&1&a\\0&0&0&1\end{pmatrix},
$$

extended by the identity on any further columns. $\mathbf{A}$ is invertible: $\mathbf{A}\mathbf{v}=\mathbf{0}$ gives $v_4=0$ from the last row, then $v_3=0$ and $v_2=0$, then $v_1=0$. By Proposition 3.12 the fit is unchanged.

**Step 2: the coefficients.** The fitted function $\hat\beta_0+\hat\beta_1x_1+\hat\beta_2x_2+\hat\beta_3x_1x_2$ written in the new variables is $\tilde\beta_0+\tilde\beta_1(x_1+a)+\tilde\beta_2(x_2+b)+\tilde\beta_3(x_1+a)(x_2+b)$; expanding and matching the coefficients of $x_1x_2$, $x_1$ and $x_2$ gives $\tilde\beta_3=\hat\beta_3$, $\tilde\beta_1+b\tilde\beta_3=\hat\beta_1$ and $\tilde\beta_2+a\tilde\beta_3=\hat\beta_2$, and these are the least squares coefficients by $\hat{\tilde{\boldsymbol{\beta}}}=\mathbf{A}^{-1}\hat{\boldsymbol{\beta}}$. The row of $\mathbf{A}$ for the interaction is the corresponding coordinate vector $\mathbf u'$, so $\mathbf u'\mathbf{A}^{-1}=\mathbf u'$, and with $\tilde{\mathbf{X}}=\mathbf{X}\mathbf{A}$, $\mathbf u'(\tilde{\mathbf{X}}'\tilde{\mathbf{X}})^{-1}\mathbf u=\mathbf u'\mathbf{A}^{-1}(\mathbf{X}'\mathbf{X})^{-1}\mathbf{A}'^{-1}\mathbf u =\mathbf u'(\mathbf{X}'\mathbf{X})^{-1}\mathbf u$; $s$ is the same, so the interaction's standard error and $t$ ratio agree.

**Step 3: without main effects.** The shifted column differs from $\mathbf{x}_1\mathbf{x}_2$ by $b\mathbf{x}_1+a\mathbf{x}_2+ab\mathbf{1}$, so the two column spaces coincide only if $b\mathbf{x}_1+a\mathbf{x}_2$ lies in $\mathcal{C}([\mathbf{1}\ \ \mathbf{x}_1\mathbf{x}_2])$. ∎

**Remark** (ISLR §3.3.2). So an interaction model with its main effects makes the same fit whatever the origins of $x_1$ and $x_2$, and only the interaction coefficient is interpretable without reference to them; the main-effect coefficients are the slopes at $x_2=0$ and at $x_1=0$. Dropping a main effect because its $t$ ratio is small makes the fit depend on an arbitrary origin. This is the content of the **hierarchical principle**: if an interaction is included, its main effects are included whatever their $p$-values.

**Example 3.6** (Synergy between media; ISLR §3.3.2).

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
adv <- read.csv("data/islr/Advertising.csv")[, -1]
fit_add <- lm(sales ~ TV + radio, data = adv)
fit_int <- lm(sales ~ TV * radio, data = adv)
round(coef(summary(fit_int)), 5)
R2a <- summary(fit_add)$r.squared; R2i <- summary(fit_int)$r.squared
c(R2_additive = R2a, R2_interaction = R2i, share_of_remaining = (R2i - R2a) / (1 - R2a))

The interaction has $t=20.7$: the relation is not additive. $R^2$ rises from $89.7\%$ to $96.8\%$, so the interaction explains $69\%$ of the variability left by the additive model. By Proposition 3.17, a further \$1,000 of television advertising is associated with $(\hat\beta_1+\hat\beta_3\,\texttt{radio})\times1000 =19+1.1\,\texttt{radio}$ units of sales, and a further \$1,000 of radio with $29+1.1\,\texttt{TV}$ units.

**Example 3.7** (Student status and income; ISLR §3.3.2).

In [ ]:
fit_cs <- lm(Balance ~ Income + Student, data = Credit)
fit_csi <- lm(Balance ~ Income * Student, data = Credit)
round(coef(summary(fit_cs)), 4)
round(coef(summary(fit_csi)), 4)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.6)
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.4, 1.4, 0.6), mgp = c(1.9, 0.6, 0))
for (m in list(fit_cs, fit_csi)) {
  plot(Credit$Income, Credit$Balance, pch = 16, cex = 0.35, col = "grey55",
       xlab = "Income", ylab = "Balance", main = if (identical(m, fit_cs)) "no interaction" else "interaction",
       cex.main = 0.8)
  xs <- seq(0, 190, length.out = 100)
  lines(xs, predict(m, data.frame(Income = xs, Student = "Yes")), col = "#B45309", lwd = 2)
  lines(xs, predict(m, data.frame(Income = xs, Student = "No")), col = "steelblue", lwd = 2)
}
legend("topleft", c("student", "non-student"), col = c("#B45309", "steelblue"), lwd = 2,
       bty = "n", cex = 0.7)
par(op)

*Figure 3.3.* Balance against income with separate lines for students and non-students, without and with the interaction (ISLR Figure 3.7).

Without the interaction the lines are parallel, $382.67$ apart. With it the student line has slope $6.218-1.999=4.219$ against $6.218$ for non-students; the difference in slopes has $t=-1.15$, $p=0.25$, so the data do not establish it.

**Example 3.8** (Term life insurance, continued: an interaction with a binary variable; FREES §3.5.3).

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
fit_si <- lm(LNFACE ~ LNINCOME * SINGLE, data = term)
round(coef(summary(fit_si)), 4)
b_si <- coef(fit_si)
rbind(other  = c(intercept = b_si[[1]], slope = b_si[[2]]),
      single = c(intercept = b_si[[1]] + b_si[[3]], slope = b_si[[2]] + b_si[[4]]))

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.6)
op <- par(mfrow = c(1, 2), mar = c(3.2, 3.4, 1.4, 0.6), mgp = c(1.9, 0.6, 0))
for (m in list(fit_single, fit_si)) {
  plot(term$LNINCOME, term$LNFACE, pch = ifelse(term$SINGLE == 1, "S", "o"), cex = 0.5,
       col = ifelse(term$SINGLE == 1, "#B45309", "grey45"), xlab = "LNINCOME", ylab = "LNFACE",
       main = if (identical(m, fit_single)) "SINGLE additive" else "SINGLE x LNINCOME", cex.main = 0.8)
  xs <- seq(min(term$LNINCOME), max(term$LNINCOME), length.out = 100)
  lines(xs, predict(m, data.frame(LNINCOME = xs, SINGLE = 1)), col = "#B45309", lwd = 2)
  lines(xs, predict(m, data.frame(LNINCOME = xs, SINGLE = 0)), col = "steelblue", lwd = 2)
}
par(op)

*Figure 3.4.* Letter plots of `LNFACE` against `LNINCOME`, “S” for single and “o” for other, with the fitted lines of the additive model and of the interaction model (FREES Figures 3.5 and 3.7).

Other respondents have the fitted line $5.78+0.573\,\texttt{LNINCOME}$, and single respondents have $-1.51+1.185\,\texttt{LNINCOME}$; the difference in slopes has $t=2.38$.

**Example 3.9** (Life insurance company expenses; FREES §3.5.3). Segal (2002) regressed the logarithm of company expenses, divided by the price of materials, on five logarithmic outputs $x_1,\ldots,x_5$, their squares, their interactions with an indicator $D$ of a branch distribution system, and the logarithmic prices of labour and capital. For the number of policies issued, $x_1$, the marginal effect is $\partial\operatorname{\mathsf{E}}[y]/\partial x_1=\beta_1+\beta_6D+2\beta_{11}x_1+2\beta_{16}Dx_1$, estimated from FREES Table 3.10 as $-0.454+0.152D+(0.064-0.014D)x_1$. At the median, $15{,}944$ policies, and with the median cost \$15,992,000:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
D <- c(baseline = 0, branch = 1)
elasticity <- -0.454 + 0.152 * D + (0.064 - 0.014 * D) * log(15944)
round(rbind(elasticity = elasticity, marginal_cost = elasticity * 15992000 / 15944), 3)

Both $y$ and $x_1$ are logarithms, so the marginal effect is an elasticity: $0.165$ for baseline and $0.182$ for branch companies. For branch companies, $\partial\,\mathrm{COST}/\partial\,\mathrm{NUMPOL}\approx0.182\,\mathrm{COST}/\mathrm{NUMPOL}$, a marginal cost of \$182.39 per policy at the medians; FREES's \$182.55 uses the elasticity rounded to $0.182$.

**Remark** (FREES §3.5.3). With several explanatory variables the second-order response function adds all squares and pairwise products, $\beta_0+\sum_j\beta_jx_j+\sum_j\beta_{jj}x_j^2+\sum_{j<l}\beta_{jl}x_jx_l$; third- and higher-order models are rarely used.

## 5 Interpreting Regression Results

> **Reading.** FREES §6.1, §6.2 (and §6.8.1, beyond the syllabus) · ISLR §3.4 · CLM §8. SRM learning outcomes 2(c), 2(f).

### 5.1 Significance and causation

**Definition 3.19** (Substantive and statistical significance; FREES §6.1.1). An effect is **statistically significant** when the test of $H_0\colon\beta_j=0$ rejects, and **substantively significant** when its size matters in the field of inquiry, in actuarial work usually financially.

**Remark** (FREES §6.1.1). The two are distinct. By Corollary 3.6, an effect can fail the test because $s$ is large, because $\mathrm{VIF}_j$ is large, or because $n$ is small, and the remedies are a more precise response, a design in which the explanatory variables are less redundant, and more data. Conversely a large sample can make an effect of no practical importance significant, and an analyst may omit it when it contradicts accepted theory. Coefficients of continuous variables are best reported per meaningful change of $x$, with a standard error or confidence interval.

**Remark** (Causal effects; FREES §6.1.1). Three conditions are necessary for a causal reading of an association: the association itself, the right time order, and the elimination of alternative explanations or an established causal mechanism. Data supply the first, sometimes the second, and never the third on their own: in Galton's data the parents' heights precede the children's and are strongly associated with them, and a family diet affecting both remains possible.

**Example 3.10** (Race, redlining and automobile insurance prices; FREES §6.1.1). Harrington and Niehaus (1998) regressed the logarithm of the loss ratio, claims over premiums, of comprehensive coverage on the proportion of Black residents (`PCTBLACK`) and controls, for $270$ Missouri ZIP codes around six cities; the data are not distributed. Discriminatory pricing would lower the loss ratio where minorities are concentrated, a negative coefficient. FREES Table 6.1 reports $0.11$ for `PCTBLACK` with $t=0.63$, significant coefficients only for the logarithms of total population ($t=-4.43$) and of average car value ($t=-3.26$), and $R_a^2=0.11$. The estimate is positive and insignificant, which is evidence against discrimination in pricing of the size the study could detect, not proof of its absence; the authors report that other specifications, weighted and unweighted, agree.

### 5.2 Omitted and extraneous variables

**Theorem 3.20** (Omitted variables; FREES §6.2.2 and, beyond the syllabus, §6.8.1; CLM Theorem 8.3, Remark 8.5, Theorem 8.6). Let $\mathbf{Y}=\mathbf{X}_1\boldsymbol{\beta}_1+\mathbf{X}_2\boldsymbol{\beta}_2+\boldsymbol{\varepsilon}$ with A1–A4 for $\mathbf{X}=[\mathbf{X}_1\ \mathbf{X}_2]$, and let only $\mathbf{X}_1$, with $p$ columns, be fitted: $\mathbf b_1=(\mathbf{X}_1'\mathbf{X}_1)^{-1}\mathbf{X}_1'\mathbf{Y}$, fitted values $\hat{\mathbf{Y}}_1=\mathbf{H}_1\mathbf{Y}$, and $s_1^2=\mathrm{SSE}_1/(n-p)$ with $\mathrm{SSE}_1=\mathbf{Y}'\mathbf{M}_1\mathbf{Y}$. Then

(i) $\operatorname{\mathsf{E}}[\mathbf b_1]=\boldsymbol{\beta}_1+(\mathbf{X}_1'\mathbf{X}_1)^{-1}\mathbf{X}_1'\mathbf{X}_2\,\boldsymbol{\beta}_2$;

(ii) the fitted values have bias $\operatorname{\mathsf{E}}[\hat{\mathbf{Y}}_1]-\operatorname{\mathsf{E}}[\mathbf{Y}]=-\mathbf{M}_1\mathbf{X}_2\boldsymbol{\beta}_2$;

(iii) $\operatorname{\mathsf{E}}[s_1^2]=\sigma^2+\dfrac{\|\mathbf{M}_1\mathbf{X}_2\boldsymbol{\beta}_2\|^2}{n-p} =\sigma^2+\dfrac{1}{n-p}\sum_{i=1}^n\mathrm{Bias}(\hat Y_{1i})^2\geqslant\sigma^2$, with equality iff $\mathbf{X}_2\boldsymbol{\beta}_2\in\mathcal{C}(\mathbf{X}_1)$;

(iv) in every sample, $\mathbf b_1=\hat{\boldsymbol{\beta}}_1+(\mathbf{X}_1'\mathbf{X}_1)^{-1}\mathbf{X}_1'\mathbf{X}_2\,\hat{\boldsymbol{\beta}}_2$, where $(\hat{\boldsymbol{\beta}}_1',\hat{\boldsymbol{\beta}}_2')'$ are the coefficients of the regression on $\mathbf{X}$.

**Proof.** **(i)** Substituting the true $\mathbf{Y}$,

$$
\mathbf b_1=(\mathbf{X}_1'\mathbf{X}_1)^{-1}\mathbf{X}_1'(\mathbf{X}_1\boldsymbol{\beta}_1+\mathbf{X}_2\boldsymbol{\beta}_2+\boldsymbol{\varepsilon})
=\boldsymbol{\beta}_1+(\mathbf{X}_1'\mathbf{X}_1)^{-1}\mathbf{X}_1'\mathbf{X}_2\boldsymbol{\beta}_2+(\mathbf{X}_1'\mathbf{X}_1)^{-1}\mathbf{X}_1'\boldsymbol{\varepsilon} ,
$$

and $\operatorname{\mathsf{E}}[\boldsymbol{\varepsilon}]=\mathbf{0}$ by A2.

**(ii)** $\operatorname{\mathsf{E}}[\mathbf{H}_1\mathbf{Y}]=\mathbf{H}_1(\mathbf{X}_1\boldsymbol{\beta}_1+\mathbf{X}_2\boldsymbol{\beta}_2)=\mathbf{X}_1\boldsymbol{\beta}_1+\mathbf{H}_1\mathbf{X}_2\boldsymbol{\beta}_2$, using $\mathbf{H}_1\mathbf{X}_1=\mathbf{X}_1$; subtracting $\operatorname{\mathsf{E}}[\mathbf{Y}]=\mathbf{X}_1\boldsymbol{\beta}_1+\mathbf{X}_2\boldsymbol{\beta}_2$ leaves $-(\mathbf{I}-\mathbf{H}_1)\mathbf{X}_2\boldsymbol{\beta}_2=-\mathbf{M}_1\mathbf{X}_2\boldsymbol{\beta}_2$.

**(iii)** By the prerequisites' lemma **expectation of a quadratic form** with mean $\mathbf{X}\boldsymbol{\beta}$ and variance $\sigma^2\mathbf{I}_n$, $\operatorname{\mathsf{E}}[\mathbf{Y}'\mathbf{M}_1\mathbf{Y}]=\sigma^2\operatorname{tr}(\mathbf{M}_1)+(\mathbf{X}_1\boldsymbol{\beta}_1+\mathbf{X}_2\boldsymbol{\beta}_2)'\mathbf{M}_1(\mathbf{X}_1\boldsymbol{\beta}_1+\mathbf{X}_2\boldsymbol{\beta}_2)$. Here $\operatorname{tr}(\mathbf{M}_1)=n-p$ by Unit 2's theorem **properties of $\mathbf{H}$ and $\mathbf{M}$**, and $\mathbf{M}_1\mathbf{X}_1=\mathbf{0}$ leaves $\boldsymbol{\beta}_2'\mathbf{X}_2'\mathbf{M}_1\mathbf{X}_2\boldsymbol{\beta}_2=\|\mathbf{M}_1\mathbf{X}_2\boldsymbol{\beta}_2\|^2$, $\mathbf{M}_1$ being symmetric and idempotent. Dividing by $n-p$ gives the first expression, and by (ii) $\|\mathbf{M}_1\mathbf{X}_2\boldsymbol{\beta}_2\|^2$ is the sum of the squared biases. It vanishes iff $\mathbf{M}_1\mathbf{X}_2\boldsymbol{\beta}_2=\mathbf{0}$, that is iff $\mathbf{X}_2\boldsymbol{\beta}_2=\mathbf{H}_1\mathbf{X}_2\boldsymbol{\beta}_2\in\mathcal{C}(\mathbf{X}_1)$.

**(iv)** The first block of the normal equations of the regression on $\mathbf{X}$ is $\mathbf{X}_1'\mathbf{X}_1\hat{\boldsymbol{\beta}}_1+\mathbf{X}_1'\mathbf{X}_2\hat{\boldsymbol{\beta}}_2=\mathbf{X}_1'\mathbf{Y}$, and $\mathbf{X}_1'\mathbf{Y}=\mathbf{X}_1'\mathbf{X}_1\mathbf b_1$ by the normal equations of the short regression. Multiplying by $(\mathbf{X}_1'\mathbf{X}_1)^{-1}$ gives the identity. ∎

**Remark** (CLM Remark 8.4). The bias in (i) is the matrix of coefficients of the regressions of the omitted columns on the included ones, times the omitted effects: it vanishes when $\mathbf{X}_1'\mathbf{X}_2=\mathbf{0}$ or $\boldsymbol{\beta}_2=\mathbf{0}$. With one included explanatory variable, an intercept, and one omitted variable, the bias of the slope has the sign of $\beta_2$ times the sign of the sample correlation of the two variables, by Unit 2's corollary **the fitted line**.

**Proposition 3.21** (Extraneous variables; FREES §6.2.1 and, beyond the syllabus, §6.8.1; CLM Proposition 8.7). Let the true model be $\mathbf{Y}=\mathbf{X}_1\boldsymbol{\beta}_1+\boldsymbol{\varepsilon}$ with A1–A4 for $\mathbf{X}_1$, and let the long model $\mathbf{X}=[\mathbf{X}_1\ \mathbf{X}_2]$, of full column rank with fewer columns than $n$, be fitted, giving $(\mathbf b_1^{L\prime},\mathbf b_2^{L\prime})'$ and $s_L^2$. Then $\mathbf b_1^L$ is unbiased for $\boldsymbol{\beta}_1$, $\mathbf b_2^L$ for $\mathbf{0}$, the fitted values for $\operatorname{\mathsf{E}}[\mathbf{Y}]$ and $s_L^2$ for $\sigma^2$, and $\operatorname{var}(\mathbf b_1^L)-\operatorname{var}(\mathbf b_1)$ is positive semi-definite, $\mathbf b_1$ being the estimator of the true model.

**Proof.** The true model is the long model with $\boldsymbol{\beta}_2=\mathbf{0}$, and A1–A4 hold for it with design $\mathbf{X}$. Unit 2's theorem **unbiasedness** gives $\operatorname{\mathsf{E}}[\mathbf b^L]=(\boldsymbol{\beta}_1',\mathbf{0}')'$, hence $\operatorname{\mathsf{E}}[\mathbf{X}\mathbf b^L]=\mathbf{X}_1\boldsymbol{\beta}_1=\operatorname{\mathsf{E}}[\mathbf{Y}]$, and Unit 2's theorem **unbiasedness of the mean square error** gives $\operatorname{\mathsf{E}}[s_L^2]=\sigma^2$. The estimator $\mathbf b_1^L$ consists of the first $p$ rows of $(\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\mathbf{Y}$, so it is $\mathbf{C}\mathbf{Y}$ for a constant $\mathbf{C}$ and is unbiased for $\boldsymbol{\beta}_1$ whatever $\boldsymbol{\beta}_1$. Unit 2's theorem **Gauss–Markov** for the true model says that $\mathbf b_1$ has the smallest variance matrix among such estimators, so $\operatorname{var}(\mathbf b_1^L)-\operatorname{var}(\mathbf b_1)$ is positive semi-definite. ∎

**Remark** (FREES §6.2). Omitting a variable that matters biases the coefficients and the fitted values, inflates $\operatorname{\mathsf{E}}[s^2]$ by $\sum_i\mathrm{Bias}(\hat Y_{1i})^2/(n-p)$ (Theorem 3.20(iii)), and with it the width of prediction intervals; including one that does not matter keeps everything unbiased and costs only precision (Proposition 3.21). Underfitting is the more serious error, hence FREES's advice: when in doubt, leave the variable in.

**Remark** (Parsimony; FREES §6.2). Selection is nonetheless guided by **parsimony**, or Occam's razor: of several explanations, use the simplest. FREES gives three reasons to prefer a simpler model: it is easier to interpret; parsimonious models often fit out-of-sample data well; and extraneous variables can bring collinearity, which makes individual coefficients hard to interpret. The counterweight is “the simplest model possible, but no simpler”, and the remark above says which way to err.

### 5.3 Other interpretations

**Remark** (Pricing, benchmarking, prediction; FREES §6.1.2). With claims as the response, the regression function $\operatorname{\mathsf{E}}[y\mid\mathbf{x}]$ is the basis of a pricing function, and its shape and its levels at key combinations of $\mathbf{x}$ are the object. In a benchmarking study the object is one observation: its residual, standardised by $s\sqrt{1-h_{ii}}$ (Unit 2), measures how far it lies from what its explanatory variables lead one to expect, without saying why. A prediction interval (Unit 2) assumes that the new observation follows the model of the sample and that $\mathbf{x}_0$ lies among the sampled $\mathbf{x}_i$: its width grows with $h_0=\mathbf{x}_0'(\mathbf{X}'\mathbf{X})^{-1}\mathbf{x}_0$, which for $k=1$ is $1/n+(x_0-\bar x)^2/\{(n-1)s_x^2\}$ (Unit 2) and grows with $|x_0-\bar x|$, and far outside the sampled range the model has no support at all, as with a lottery prediction for a population ten times the largest sampled.

**Example 3.11** (The marketing plan; ISLR §3.4). ISLR's seven questions about the `Advertising` data, answered from the fits of Unit 2 and of Example 3.6.

In [ ]:
fit_all <- lm(sales ~ TV + radio + newspaper, data = adv)
sm_all <- summary(fit_all)
c(F = sm_all$fstatistic[["value"]],
  p_F = pf(sm_all$fstatistic[["value"]], 3, 196, lower.tail = FALSE),
  RSE = sm_all$sigma, mean_sales = mean(adv$sales), RSE_over_mean = sm_all$sigma / mean(adv$sales),
  R2 = sm_all$r.squared)
round(cbind(coef(sm_all)[, c(1, 4)], confint(fit_all)), 4)
X_m <- adv[, c("TV", "radio", "newspaper")]
round(sapply(names(X_m), function(v)
  1 / (1 - summary(lm(as.formula(paste(v, "~ .")), data = X_m))$r.squared)), 3)
fit_q <- lm(sales ~ TV * radio + I(TV^2), data = adv)
round(coef(summary(fit_q)), 6)
c(R2_with_TV_squared = summary(fit_q)$r.squared)

(1) **Is there a relation between sales and advertising?** The overall $F$ is $570$ with $p$-value $1.6\times10^{-96}$. (2) **How strong?** The residual standard error, $1.69$, is $12\%$ of the mean sales, $14.02$, and $R^2=0.897$. (3) **Which media?** Television and radio have $p$-values below $10^{-4}$; newspaper has $p=0.86$. (4) **How large is each association?** The $95\%$ intervals are $(0.043,0.049)$ for `TV`, $(0.172,0.206)$ for `radio` and $(-0.013,0.011)$ for `newspaper`; by Proposition 3.1 only the last contains $0$. The variance inflation factors of Corollary 3.6 are $1.005$, $1.145$ and $1.145$, so the width of the newspaper interval is not due to redundancy among the media. (5) **How accurately can future sales be predicted?** By Unit 2's confidence and prediction intervals, the first for average sales at given budgets, the second, always wider, for one market. (6) **Is the relation linear?** No: added to the model of Example 3.6, the square of `TV` has $t=-15.9$, whose square is its partial $F$ (Corollary 3.9), and $R^2$ rises to $0.986$; the negative coefficient means that sales rise with television spending at a decreasing rate. (7) **Is there synergy?** Yes: in Example 3.6 the interaction of `TV` and `radio` has $t=20.7$ and raises $R^2$ from $0.897$ to $0.968$.

## 6 Computation in R

> **Reading.** ISLR §3.6.4–3.6.7 · FREES chapter 3 script.

### 6.1 The ISLR lab

`lstat:age` enters the product of two variables and `lstat * age` enters `lstat`, `age` and `lstat:age`.

In [ ]:
summary(lm(medv ~ lstat * age, data = Boston))$coefficients

`I(lstat\textasciicircum2)` enters the square, the operator `\textasciicircum` having another meaning inside a formula; `anova` of two nested fits is the partial $F$ test of Corollary 3.9.

In [ ]:
lm.fit <- lm(medv ~ lstat, data = Boston)
lm.fit2 <- lm(medv ~ lstat + I(lstat^2), data = Boston)
summary(lm.fit2)$coefficients
anova(lm.fit, lm.fit2)
c(t_squared = coef(summary(lm.fit2))[3, "t value"]^2)

The quadratic term has $F=135$, the square of its $t$ ratio, with $p$-value $0$ to machine precision.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3.3)
op <- par(mfrow = c(2, 2), mar = c(3.2, 3.4, 1.6, 0.6), mgp = c(1.9, 0.6, 0), cex.main = 0.8)
plot(lm.fit2, cex = 0.4)
par(op)

*Figure 3.5.* The four diagnostic plots of the quadratic fit of `medv` on `lstat`.

`poly(lstat, 5)` enters an orthogonal basis of the polynomials of degree at most $5$; its column space is that of the raw powers, so by Proposition 3.12 the fitted values agree while the coefficients differ.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
lm.fit5 <- lm(medv ~ poly(lstat, 5), data = Boston)
summary(lm.fit5)$coefficients
lm.fit5raw <- lm(medv ~ poly(lstat, 5, raw = TRUE), data = Boston)
c(fitted_differ_by = max(abs(fitted(lm.fit5) - fitted(lm.fit5raw))))
round(summary(lm(medv ~ poly(lstat, 7), data = Boston))$coefficients[7:8, 4], 4)
summary(lm(medv ~ log(rm), data = Boston))$coefficients

All five polynomial terms are significant, and in a seventh-degree fit the sixth and seventh are not ($p=0.21$ and $0.71$). The `Carseats` data: unit sales of child car seats at $400$ stores with the qualitative `ShelveLoc` (Bad, Medium, Good), for which `lm` creates the indicators shown by `contrasts`, Bad being the baseline.

In [ ]:
lm.fitc <- lm(Sales ~ . + Income:Advertising + Price:Age, data = Carseats)
round(summary(lm.fitc)$coefficients, 4)
contrasts(Carseats$ShelveLoc)

A good shelving location is associated with $4.85$ thousand more units than a bad one and a medium location with $1.95$ thousand more, at fixed values of the other variables.

In [ ]:
LoadLibraries <- function() {
  library(ISLR2)
  library(MASS)
  print("The libraries have been loaded.")
}
LoadLibraries()
detach("package:MASS")                 # MASS's own Boston would mask ISLR2's

### 6.2 FREES's chapter 3 script

The script's models of `LNFACE` with the marital-status indicators $\texttt{MAR0}=\texttt{SINGLE}$, `MAR1` (married) and `MAR2` (living with a partner), additive and with the interaction, with and without the other two variables:

In [ ]:
term$MAR1 <- as.numeric(term$MARSTAT == 1); term$MAR2 <- as.numeric(term$MARSTAT == 2)
one_row <- function(f, v) coef(summary(lm(f, data = term)))[v, c(1, 3)]
round(rbind(
  MAR0             = one_row(LNFACE ~ LNINCOME + SINGLE, "SINGLE"),
  MAR1             = one_row(LNFACE ~ LNINCOME + MAR1, "MAR1"),
  MAR2             = one_row(LNFACE ~ LNINCOME + MAR2, "MAR2"),
  MAR0_with_EDU_HH = one_row(LNFACE ~ LNINCOME + SINGLE + EDUCATION + NUMHH, "SINGLE"),
  MAR0xLNINCOME    = one_row(LNFACE ~ LNINCOME * SINGLE + EDUCATION + NUMHH, "LNINCOME:SINGLE")), 4)

With education and household size in the model the `SINGLE` coefficient moves from $-0.800$ to $-0.511$. By Theorem 3.20(iv) the difference is exactly the coefficients of the regressions of the two added variables on `LNINCOME` and `SINGLE`, times their coefficients in the long regression:

In [ ]:
fit_short <- lm(LNFACE ~ LNINCOME + SINGLE, data = term)
fit_long <- lm(LNFACE ~ LNINCOME + SINGLE + EDUCATION + NUMHH, data = term)
aux <- coef(lm(cbind(EDUCATION, NUMHH) ~ LNINCOME + SINGLE, data = term))
round(aux, 4)
round(cbind(short = coef(fit_short), long = coef(fit_long)[1:3],
            difference = coef(fit_short) - coef(fit_long)[1:3],
            aux_times_long = drop(aux %*% coef(fit_long)[4:5])), 5)
round(aux["SINGLE", ] * coef(fit_long)[4:5], 5)
round(coef(fit_long)[4:5], 5)

At fixed income a single respondent's household has $1.61$ fewer members, each worth $0.2475$ in logarithmic face amount, which accounts for $-0.399$ of the difference; education contributes $+0.110$.

### 6.3 Formulas against package output

The unrestricted fit is `medv` on `lstat`, `rm`, `age`, `indus` and `tax`, and $H_0$ sets the last three coefficients to zero.

In [ ]:
fit_U <- lm(medv ~ lstat + rm + age + indus + tax, data = Boston)
fit_R <- lm(medv ~ lstat + rm, data = Boston)
X <- model.matrix(fit_U); n <- nrow(X); k <- ncol(X) - 1; b <- coef(fit_U)
R <- cbind(matrix(0, 3, 3), diag(3)); q <- nrow(R); s2 <- summary(fit_U)$sigma^2
XtXi <- solve(crossprod(X))
F_quad <- drop(t(R %*% b) %*% solve(R %*% XtXi %*% t(R)) %*% (R %*% b)) / (q * s2)
SSE_U <- sum(residuals(fit_U)^2); SSE_R <- sum(residuals(fit_R)^2)
F_sse <- ((SSE_R - SSE_U) / q) / (SSE_U / (n - k - 1))
c(F_quadratic_form = F_quad, F_extra_SS = F_sse, F_anova = anova(fit_R, fit_U)$F[2],
  p = pf(F_sse, q, n - k - 1, lower.tail = FALSE))

Restricted least squares for $H_0\colon\beta_{\texttt{age}}=\beta_{\texttt{indus}}$, a constraint that is not an exclusion, against the refit on the sum of the two variables:

In [ ]:
R1 <- matrix(c(0, 0, 0, 1, -1, 0), 1)
b_R <- drop(b - XtXi %*% t(R1) %*% solve(R1 %*% XtXi %*% t(R1)) %*% (R1 %*% b))
fit_sum <- lm(medv ~ lstat + rm + I(age + indus) + tax, data = Boston)
gap_formula <- drop(t(R1 %*% b) %*% solve(R1 %*% XtXi %*% t(R1)) %*% (R1 %*% b))
c(max_coef_diff = max(abs(b_R[c(1:3, 6)] - coef(fit_sum)[c(1:3, 5)])),
  b_age = b_R[[4]], b_indus = b_R[[5]], b_sum = coef(fit_sum)[[4]],
  gap_formula = gap_formula, gap_refit = sum(residuals(fit_sum)^2) - SSE_U)

Frisch–Waugh–Lovell for `rm`, the partial correlation identity for every slope, and the variance inflation factors of Corollary 3.6 against `car::vif`:

In [ ]:
e_y <- residuals(lm(medv ~ lstat + age + indus + tax, data = Boston))
e_x <- residuals(lm(rm ~ lstat + age + indus + tax, data = Boston))
c(slope_partialled = unname(coef(lm(e_y ~ e_x - 1))), b_rm = b[["rm"]],
  max_resid_diff = max(abs(residuals(lm(e_y ~ e_x - 1)) - residuals(fit_U))))
t_U <- coef(summary(fit_U))[-1, "t value"]
partial_direct <- sapply(names(t_U), function(v) {
  others <- setdiff(names(t_U), v)
  cor(residuals(lm(reformulate(others, "medv"), data = Boston)),
      residuals(lm(reformulate(others, v), data = Boston)))
})
round(rbind(direct = partial_direct, from_t = t_U / sqrt(t_U^2 + n - k - 1)), 6)
vif_R2 <- sapply(names(t_U), function(v)
  1 / (1 - summary(lm(reformulate(setdiff(names(t_U), v), v), data = Boston))$r.squared))
round(rbind(one_over_1_minus_R2 = vif_R2, car_vif = car::vif(fit_U)), 6)
se_vif <- sqrt(s2) * sqrt(vif_R2) / (sapply(Boston[names(t_U)], sd) * sqrt(n - 1))
round(rbind(formula = se_vif, lm = coef(summary(fit_U))[-1, "Std. Error"]), 6)

Changing the baseline of `ShelveLoc`, and centring the two variables of an interaction, against Propositions 3.12 and 3.18:

In [ ]:
cs_bad <- lm(Sales ~ Price + ShelveLoc, data = Carseats)
cs_good <- lm(Sales ~ Price + relevel(ShelveLoc, ref = "Good"), data = Carseats)
c(fitted_diff = max(abs(fitted(cs_bad) - fitted(cs_good))),
  F_bad = anova(lm(Sales ~ Price, data = Carseats), cs_bad)$F[2],
  F_good = anova(lm(Sales ~ Price, data = Carseats), cs_good)$F[2])
rbind(bad_baseline = coef(cs_bad), good_baseline = coef(cs_good))
Bc <- transform(Boston, lstat = lstat - mean(lstat), age = age - mean(age))
h_raw <- lm(medv ~ lstat * age, data = Boston); h_cen <- lm(medv ~ lstat * age, data = Bc)
rbind(raw = coef(summary(h_raw))[, 3], centred = coef(summary(h_cen))[, 3])
c(fitted_diff = max(abs(fitted(h_raw) - fitted(h_cen))),
  no_main_raw_SSE = sum(residuals(lm(medv ~ lstat:age, data = Boston))^2),
  no_main_centred_SSE = sum(residuals(lm(medv ~ lstat:age, data = Bc))^2))

The indicator coefficients change with the baseline while the fitted values and the $F$ test for `ShelveLoc` do not. Centring `lstat` and `age` leaves the interaction's $t$ ratio and the fitted values unchanged and moves the main-effect $t$ ratios; without the main effects, centring changes $\mathrm{SSE}$.

## 7 Exercises

**Exercise 3.1** (FREES exercise 3.4, parts (g), (h)). In the regression of `SALES` on `MEDSCHYR`, `MEDHVL` and `POP` for the $50$ ZIP codes of the Wisconsin lottery data, the coefficient of `MEDSCHYR` is negative. (g) Compute the partial correlation coefficient of `SALES` and `MEDSCHYR`, and interpret it. (h) Produce the added variable plot of `SALES` against `MEDSCHYR` controlling for `MEDHVL` and `POP`, and check that its correlation agrees with (g).

**Solution.**

In [ ]:
lot <- read.csv("data/frees/csv/WiscLottery.csv")
fit_l3 <- lm(SALES ~ MEDSCHYR + MEDHVL + POP, data = lot)
t_ms <- coef(summary(fit_l3))["MEDSCHYR", "t value"]
e_sales <- residuals(lm(SALES ~ MEDHVL + POP, data = lot))
e_school <- residuals(lm(MEDSCHYR ~ MEDHVL + POP, data = lot))
c(t = t_ms, partial_from_t = t_ms / sqrt(t_ms^2 + 50 - 4),
  partial_direct = cor(e_sales, e_school))
round(cor(lot[, c("SALES", "MEDSCHYR", "MEDHVL", "POP")]), 3)
c(e_school_obs11 = e_school[[11]], largest_other = max(abs(e_school[-11])))
t49 <- coef(summary(update(fit_l3, data = lot[-11, ])))["MEDSCHYR", "t value"]
c(t_without_11 = t49, partial_without_11 = t49 / sqrt(t49^2 + 49 - 4))

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.6)
op <- par(mar = c(3.2, 3.4, 1.2, 0.6), mgp = c(1.9, 0.6, 0))
plot(e_school, e_sales, pch = 16, cex = 0.6, xlab = "MEDSCHYR, residual on MEDHVL and POP",
     ylab = "SALES, residual")
abline(lm(e_sales ~ e_school)); text(e_school[11], e_sales[11], "11", pos = 2, cex = 0.7)
par(op)

(g) By Proposition 3.5 with $t=-2.83$ and $n-(k+1)=46$, $r=-2.83/\sqrt{2.83^2+46}=-0.385$. With home value and population held fixed, ZIP codes with more schooling have lower sales, although the simple correlation of `SALES` and `MEDSCHYR` is $+0.467$: schooling is correlated with home value ($0.708$) and population ($0.590$), which carry the positive association. (h) The correlation in the plot is $-0.385$, the value of (g). Observation $11$, Shorewood, has a horizontal residual of $1.75$ against at most $0.79$ for the others, so it dominates the slope: refitted on the other $49$ ZIP codes, the regression gives `MEDSCHYR` the $t$ ratio $-0.81$ and the partial correlation $-0.120$. ■

**Exercise 3.2** (FREES exercise 3.5, parts (g)–(i)). In the insurer expense data, with every non-binary variable transformed to $\ln(1+x)$: (g) square the two loss and the two premium variables and fit `LNEXPENSES` on the eight variables of part (f), namely assets, `GROUP`, both losses, both premiums and the two wage variables, and the four squares; report $s$, $R^2$, $R_a^2$, and decide whether the quadratic terms are useful. (h) Omit the two wage variables and refit; compare the number of observations with part (f). (i) Drop the quadratic terms of (g) and add the interactions of `GROUP` with assets and with the four loss and premium variables, eleven explanatory variables in all; report $s$, $R^2$, $R_a^2$, and estimate the change in `EXPENSES` per dollar of `GPWCOMM`, at the medians of each `GROUP`, for $\texttt{GROUP}=0$ and $\texttt{GROUP}=1$.

**Solution.**

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
naic <- read.csv("data/frees/csv/NAICExpense.csv")
vars_n <- c("EXPENSES", "LONGLOSS", "SHORTLOSS", "GPWPERSONAL", "GPWCOMM", "ASSETS",
            "STAFFWAGE", "AGENTWAGE")
L <- as.data.frame(lapply(naic[, vars_n], function(v) log(1 + v)))
names(L) <- paste0("LN", vars_n); L$GROUP <- naic$GROUP
for (v in c("LNLONGLOSS", "LNSHORTLOSS", "LNGPWPERSONAL", "LNGPWCOMM")) L[[paste0(v, "2")]] <- L[[v]]^2
stats3 <- function(m) c(n = nobs(m), s = summary(m)$sigma, R2 = summary(m)$r.squared,
                        R2_adj = summary(m)$adj.r.squared)
f_f <- lm(LNEXPENSES ~ LNASSETS + GROUP + LNLONGLOSS + LNSHORTLOSS + LNGPWPERSONAL +
            LNGPWCOMM + LNSTAFFWAGE + LNAGENTWAGE, data = L)
f_g <- update(f_f, . ~ . + LNLONGLOSS2 + LNSHORTLOSS2 + LNGPWPERSONAL2 + LNGPWCOMM2)
f_h <- lm(LNEXPENSES ~ LNASSETS + GROUP + LNLONGLOSS + LNSHORTLOSS + LNGPWPERSONAL +
            LNGPWCOMM + LNLONGLOSS2 + LNSHORTLOSS2 + LNGPWPERSONAL2 + LNGPWCOMM2, data = L)
f_i0 <- lm(LNEXPENSES ~ LNASSETS + LNLONGLOSS + LNSHORTLOSS + LNGPWPERSONAL + LNGPWCOMM +
             GROUP, data = L)
f_i <- lm(LNEXPENSES ~ (LNASSETS + LNLONGLOSS + LNSHORTLOSS + LNGPWPERSONAL + LNGPWCOMM) *
            GROUP, data = L)
round(rbind(f = stats3(f_f), g = stats3(f_g), h = stats3(f_h), i = stats3(f_i)), 5)
round(coef(summary(f_g))[10:13, ], 4)
c(F_quadratics = anova(f_f, f_g)$F[2], p = anova(f_f, f_g)[2, "Pr(>F)"],
  F_interactions = anova(f_i0, f_i)$F[2], p = anova(f_i0, f_i)[2, "Pr(>F)"],
  AGENTWAGE_missing = sum(is.na(naic$AGENTWAGE)))
b_i <- coef(f_i)
t(sapply(0:1, function(g) {
  d <- naic[naic$GROUP == g, ]
  slope <- b_i[["LNGPWCOMM"]] + g * b_i[["LNGPWCOMM:GROUP"]]
  c(GROUP = g, slope = slope, med_GPWCOMM = median(d$GPWCOMM), med_EXPENSES = median(d$EXPENSES),
    dollars_per_dollar = slope * (1 + median(d$EXPENSES)) / (1 + median(d$GPWCOMM)))
}))

(g) With the squares, $s=0.0210$, $R^2=0.949$ and $R_a^2=0.947$, against $0.0224$, $0.941$ and $0.940$ without them. Individually, the squares of short-tail losses and of commercial premiums have $|t|>4$, that of long-tail losses $t=2.10$, and that of personal premiums $p=0.077$; jointly, by Corollary 3.9, $F=13.5$ on $(4,352)$ degrees of freedom, $p=3\times10^{-10}$: the quadratic terms are useful. (h) Without the wage variables, $s=0.0218$, $R^2=0.942$, $R_a^2=0.941$, on $384$ companies against $365$: `AGENTWAGE` is missing for $19$ companies, and a fit uses only complete cases. (i) With the interactions, $s=0.0232$, $R^2=0.935$ and $R_a^2=0.933$. By Proposition 3.17 the slope of `LNGPWCOMM` is $0.0956$ for $\texttt{GROUP}=0$ and $0.0956-0.0272=0.0684$ for $\texttt{GROUP}=1$; differentiating $\ln(1+E)=\cdots+b\ln(1+G)$ as in Unit 2 gives $\mathrm dE/\mathrm dG=b(1+E)/(1+G)$, which at each group's medians is $0.095$ dollars per dollar for unaffiliated and $0.066$ for affiliated companies. The five interactions are jointly insignificant, $F=0.15$, $p=0.98$, so the data do not distinguish the two slopes. ■

**Exercise 3.3** (FREES exercise 3.6, part (d)). In the regression of `LIFEEXP` on `FERTILITY`, `PUBLICEDUCATION` and $\texttt{lnHEALTH}=\ln\texttt{PRIVATEHEALTH}$, the coefficient of public education is negative although its correlation with life expectancy is positive. (i) Describe the purpose of an added variable plot and a method for producing it. (ii) Calculate the correlation corresponding to the added variable plot of `PUBLICEDUCATION` versus `LIFEEXP` controlling for `FERTILITY` and `lnHEALTH`.

**Solution.** (i) It shows the relation between the response and one explanatory variable after both have been adjusted for the others: regress `LIFEEXP` on `FERTILITY` and `lnHEALTH` and keep the residuals $\mathbf e_1$; regress `PUBLICEDUCATION` on the same two and keep the residuals $\mathbf e_2$; plot $\mathbf e_1$ against $\mathbf e_2$ (Definition 3.3). By Corollary 3.4 the line through the plot has the slope of `PUBLICEDUCATION` in the full regression.

In [ ]:
un <- read.csv("data/frees/csv/UNLifeExpectancy.csv"); un$lnHEALTH <- log(un$PRIVATEHEALTH)
d_un <- na.omit(un[, c("LIFEEXP", "FERTILITY", "PUBLICEDUCATION", "lnHEALTH")])
fit_un <- lm(LIFEEXP ~ FERTILITY + PUBLICEDUCATION + lnHEALTH, data = d_un)
t_pe <- coef(summary(fit_un))["PUBLICEDUCATION", "t value"]
c(n = nrow(d_un), t = t_pe, partial_from_t = t_pe / sqrt(t_pe^2 + nrow(d_un) - 4),
  partial_direct = cor(residuals(lm(LIFEEXP ~ FERTILITY + lnHEALTH, data = d_un)),
                       residuals(lm(PUBLICEDUCATION ~ FERTILITY + lnHEALTH, data = d_un))),
  simple = cor(d_un$LIFEEXP, d_un$PUBLICEDUCATION))

(ii) By Proposition 3.5, with $t=-0.688$ on the $152$ countries with complete data, $r=-0.688/\sqrt{0.688^2+148}=-0.056$, which the residuals reproduce. With fertility and private health spending held fixed, public education spending carries almost no linear association with life expectancy; the simple correlation, $0.074$ on the same countries, is also small. ■

**Exercise 3.4** (FREES exercise 6.1). Homeowners' insurance in $47$ Chicago ZIP codes, $1977$–$78$ (`Chicago`): the percentage minority `race`, fires per $1{,}000$ housing units `fire`, thefts per $1{,}000$ population `theft`, the percentage of housing built before $1940$ `age`, new voluntary-market policies per $100$ housing units `volact`, new FAIR-plan (involuntary-market) policies `involact`, and median family `income`. (a) Summary statistics and skewness. (b) A scatterplot matrix of `volact`, `involact` and `race`; are the patterns consistent with racial discrimination? (c) The correlation matrix. (d) Fit `volact` on `race`, `fire`, `theft`, `age` and `income`: (i) comment on the coefficient of `race`; (ii) two ZIP codes have high leverage: refit without them and compare. (e) Repeat (d) for `involact`. (f) Repeat (d) for $\texttt{proportion}=\texttt{involact}/(\texttt{volact}+\texttt{involact})$. (g) Why do the same two ZIP codes have high leverage in (d), (e) and (f)? (h) What omitted variables might affect both availability and race? (i) How could the dependence of neighbouring ZIP codes be incorporated?

**Solution.**

In [ ]:
chi <- read.csv("data/frees/csv/Chicago.csv")
skew <- function(v) mean((v - mean(v))^3) / mean((v - mean(v))^2)^1.5
round(t(sapply(chi[, -1], function(v) c(mean = mean(v), median = median(v), sd = sd(v),
                                        min = min(v), max = max(v), skew = skew(v)))), 3)
round(cor(chi[, -1]), 3)

In [ ]:
pairs(chi[, c("volact", "involact", "race")], pch = 16, cex = 0.5, gap = 0.3)

In [ ]:
chi$proportion <- chi$involact / (chi$volact + chi$involact)
fit_chi <- function(y, d) lm(reformulate(c("race", "fire", "theft", "age", "income"), y), data = d)
m_v <- fit_chi("volact", chi)
lev <- hatvalues(m_v); hi <- order(lev, decreasing = TRUE)[1:2]
cbind(chi[hi, c("zipcode", "race", "fire", "theft", "income")], leverage = round(lev[hi], 3))
c(flag = 2 * 6 / 47,
  theft_z = (chi$theft[hi[1]] - mean(chi$theft)) / sd(chi$theft),
  income_z = (chi$income[hi[2]] - mean(chi$income)) / sd(chi$income))
race_row <- function(y) {
  a <- coef(summary(fit_chi(y, chi)))["race", ]; b <- coef(summary(fit_chi(y, chi[-hi, ])))["race", ]
  c(b_all = a[[1]], t_all = a[[3]], b_without = b[[1]], t_without = b[[3]])
}
round(rbind(volact = race_row("volact"), involact = race_row("involact"),
            proportion = race_row("proportion")), 4)
round(coef(summary(m_v))[, 3], 2)
round(coef(summary(fit_chi("volact", chi[-hi, ])))[, 3], 2)
sapply(c("involact", "proportion"), function(y) order(hatvalues(fit_chi(y, chi)), decreasing = TRUE)[1:2])

(a) `theft` is the most skewed variable (skewness $3.05$), then `fire` ($1.31$) and `income` ($1.19$); `age` is skewed to the left ($-0.95$). (b) Voluntary activity falls with the minority percentage ($r=-0.76$) and FAIR-plan activity rises with it ($0.71$): minority ZIP codes are served more through the involuntary market. That is what discrimination would produce, and also what higher losses in those ZIP codes would produce. (c) `race` is correlated with `fire` ($0.59$) and `income` ($-0.70$), so its simple association with availability mixes these. (d)(i) With the four controls the coefficient of `race` is $-0.060$, $t=-4.51$: at the same fire, theft, housing age and income, a ZIP code with ten more percentage points of minority residents has $0.6$ fewer voluntary policies per $100$ housing units. (ii) ZIP code $60607$ (theft $147$, $5.1$ standard deviations above the mean) and $60611$ (income \$21,480, $3.9$ standard deviations above) have leverages $0.690$ and $0.676$, far above the flag $2(k+1)/n=0.255$. Without them the `race` coefficient halves to $-0.030$ with $t=-2.23$, still significant at $5\%$; `fire` loses its significance and `income` gains it ($t=3.47$). (e) For `involact` the `race` coefficient is $0.0091$ ($t=3.94$), and $0.0096$ ($t=3.46$) without the two ZIP codes. (f) For `proportion` it is $0.0030$ ($t=4.30$), and $0.0023$ ($t=2.95$) without them. The association of race with availability survives the controls and the deletion in all three responses. (g) Leverage is a function of the explanatory variables alone (Unit 2), and the three regressions share them. (h) The data are at the ZIP-code level, and by Theorem 3.20(i) the `race` coefficient absorbs any omitted variable that affects availability and has a non-zero `race` coefficient in its regression on the included variables: the condition and value of the housing, the owners' claims and credit histories, the coverage amounts demanded, and the location of fire stations and hydrants. (i) Neighbouring ZIP codes share unmeasured conditions, so their errors are correlated, which violates A3 and invalidates $\operatorname{var}(\hat{\boldsymbol{\beta}})=\sigma^2(\mathbf{X}'\mathbf{X})^{-1}$. Region indicators or averages of the neighbours' variables can enter as explanatory variables, or the error covariance can be modelled as a decreasing function of the distance between ZIP codes. ■

**Exercise 3.5** (FREES exercise 6.2). A regression of the logarithm of the $1990$ salaries of $1{,}898$ University of Wisconsin faculty on $109$ explanatory variables ($97$ department indicators and $12$ others, FREES Tables 6.9 and 6.10) gives the coefficient $0.016$ with $t=1.86$ for $\texttt{GENDER}=1$ if male, and the ANOVA $\mathrm{SSR}=114.048$ on $109$ degrees of freedom, $\mathrm{SSE}=29.739$ on $1{,}789$; the mean salary of the men is $28\%$ above that of the women. (a) How can the study be used to evaluate the salary of a female chemist who believes she is underpaid? (b) (i) Cite statistical arguments that men are not paid significantly more than women. (ii) Cite arguments that they are. (iii) If women are paid less, by how much should their salaries be raised?

**Solution.**

In [ ]:
SSR_f <- 114.048; SSE_f <- 29.739
c(R2 = SSR_f / (SSR_f + SSE_f), s = sqrt(SSE_f / 1789), F = (SSR_f / 109) / (SSE_f / 1789),
  p_two_sided = 2 * pt(-1.86, 1789), p_one_sided = pt(-1.86, 1789),
  t_crit_two = qt(0.975, 1789), t_crit_one = qt(0.95, 1789), raise_pct = 100 * (exp(0.016) - 1))

(a) Put her rank, years, degree, cohort, market ratio and the chemistry indicator into the fitted equation to obtain her expected log salary, and compare with her actual log salary. The residual, standardised by $s\sqrt{1-h}$ with $s=0.129$, measures how far she lies from colleagues with the same characteristics (§5.3); a large negative value supports her case, without explaining it. (b)(i) The two-sided test of $H_0\colon\beta_{\texttt{GENDER}}=0$ does not reject at $5\%$: $|t|=1.86<1.96$, $p=0.063$; and the estimate, $1.6\%$, is small beside a raw difference of $28\%$. (ii) The question is directional, and the one-sided test against $\beta_{\texttt{GENDER}}>0$ rejects at $5\%$: $1.86>1.65$, $p=0.032$. Moreover rank and years at rank are among the controls; if women are promoted more slowly, part of any discrimination is carried by those variables and the `GENDER` coefficient understates it. (iii) Raising women's salaries by $100(e^{0.016}-1)=1.6\%$ removes the estimated difference at equal characteristics. The ANOVA gives $R^2=0.793$ and $F=62.9$, the $F$ that FREES prints; its total of $1{,}898$ degrees of freedom implies $1{,}899$ faculty, one more than the text states. ■

**Exercise 3.6** (ISLR §3.7, exercise 3). A model of starting salary in thousands of dollars on $X_1=\text{GPA}$, $X_2=\text{IQ}$, $X_3=\text{Level}$ ($1$ college, $0$ high school), $X_4=X_1X_2$ and $X_5=X_1X_3$ gives $\hat\beta_0=50$, $\hat\beta_1=20$, $\hat\beta_2=0.07$, $\hat\beta_3=35$, $\hat\beta_4=0.01$, $\hat\beta_5=-10$. (a) Which is correct, and why: for fixed IQ and GPA, (i) high school graduates earn more on average; (ii) college graduates earn more on average; (iii) high school graduates earn more provided the GPA is high enough; (iv) college graduates earn more provided the GPA is high enough? (b) Predict the salary of a college graduate with IQ $110$ and GPA $4.0$. (c) True or false: since the coefficient of the GPA–IQ interaction is very small, there is very little evidence of an interaction effect.

**Solution.** (a) By Proposition 3.17 with the indicator $X_3$, at fixed GPA and IQ the college graduate's expected salary exceeds the high school graduate's by $\hat\beta_3+\hat\beta_5\,\text{GPA}=35-10\,\text{GPA}$, positive for GPA below $3.5$ and negative above. So (iii) is correct; (i) and (ii) fail because the sign depends on GPA, and (iv) has the direction reversed. (b) $50+20(4)+0.07(110)+35+0.01(4)(110)-10(4)(1)=137.1$, a salary of \$137,100. (c) False. The size of a coefficient is not evidence: the evidence is its $t$ ratio, which needs the standard error. The size also depends on units: the term $0.01\,\text{GPA}\times\text{IQ}$ adds $0.01\,\text{IQ}$, $1$ thousand dollars at IQ $100$, per unit of GPA, and $4$ thousand dollars in total at GPA $4$. ■

**Exercise 3.7** (ISLR §3.7, exercise 9, parts (e), (f)). For the regression of `mpg` on the other variables of `Auto` except `name`: (e) fit models with interactions using `*` and `:`; do any appear significant? (f) Try transformations such as $\log X$, $\sqrt X$, $X^2$, and comment.

**Solution.**

In [ ]:
A <- Auto[, names(Auto) != "name"]
fit_pairs <- lm(mpg ~ .^2, data = A)
cf <- coef(summary(fit_pairs)); round(cf[cf[, 4] < 0.05, ], 4)
c(n_coef = nrow(cf), R2_main = summary(lm(mpg ~ ., data = A))$r.squared,
  R2_pairs = summary(fit_pairs)$r.squared,
  F_pairs = anova(lm(mpg ~ ., data = A), fit_pairs)$F[2],
  p_pairs = anova(lm(mpg ~ ., data = A), fit_pairs)[2, "Pr(>F)"])
round(coef(summary(lm(mpg ~ . + horsepower:weight, data = A)))["horsepower:weight", ], 6)
round(sapply(c(identity = "horsepower", log = "log(horsepower)", sqrt = "sqrt(horsepower)",
               square = "I(horsepower^2)"),
             function(f) summary(lm(as.formula(paste("mpg ~", f)), data = A))$r.squared), 4)

(e) With all $21$ pairwise interactions, the ones significant at $5\%$ are `displacement:year`, `acceleration:year` and `acceleration:origin`, and $R^2$ rises from $0.821$ to $0.889$; the $21$ interactions are jointly significant by Corollary 3.9. A single interaction such as `horsepower:weight` added to the main effects has $t=10.6$. With $21$ tests at $5\%$, about one significant interaction is expected by chance alone, so individual significance in the full model is weak evidence; the joint $F$ is the test that controls its level. (f) As the single explanatory variable for `mpg`, $\log(\texttt{horsepower})$ gives $R^2=0.668$, $\sqrt{\texttt{horsepower}}$ $0.644$, `horsepower` $0.606$ and its square $0.507$: the relation is convex, `mpg` falling with `horsepower` at a decreasing rate, as in Example 3.5. ■

**Exercise 3.8** (ISLR §3.7, exercise 10). For `Carseats`: (a) fit `Sales` on `Price`, `Urban` and `US`; (b) interpret each coefficient; (c) write the model as an equation, handling the qualitative variables properly; (d) for which predictors can $H_0\colon\beta_j=0$ be rejected? (e) fit the smaller model with only those predictors; (f) how well do the models of (a) and (e) fit? (g) give $95\%$ confidence intervals for the coefficients of (e); (h) is there evidence of outliers or high leverage observations in (e)?

**Solution.**

In [ ]:
c_a <- lm(Sales ~ Price + Urban + US, data = Carseats)
c_e <- lm(Sales ~ Price + US, data = Carseats)
round(coef(summary(c_a)), 4)
round(coef(summary(c_e)), 4)
round(rbind(a = stats3(c_a), e = stats3(c_e)), 5)
c(R2_drop = summary(c_a)$r.squared - summary(c_e)$r.squared)
c(F_Urban = anova(c_e, c_a)$F[2], t_Urban_squared = coef(summary(c_a))["UrbanYes", 3]^2)
round(confint(c_e), 4)
c(max_abs_std_resid = max(abs(rstandard(c_e))), n_above_2 = sum(abs(rstandard(c_e)) > 2),
  expected_above_2 = 400 * 2 * pnorm(-2), max_leverage = max(hatvalues(c_e)),
  mean_leverage = 3 / 400, flag = 2 * 3 / 400, n_above_flag = sum(hatvalues(c_e) > 2 * 3 / 400))

(a)–(c) $\widehat{\texttt{Sales}}=13.04-0.0545\,\texttt{Price}-0.0219\,\texttt{Urban} +1.2006\,\texttt{US}$, with `Urban` and `US` the indicators of an urban store and a US store; `Sales` is in thousands of units. A dollar more in price goes with $54.5$ fewer seats at the same location type; an urban store sells $22$ fewer seats than a rural one with the same price and country, and a US store $1{,}201$ more than a non-US one. (d) $H_0$ is rejected for `Price` ($t=-10.4$) and `US` ($t=4.63$), not for `Urban` ($p=0.94$). (e) $\widehat{\texttt{Sales}}=13.03-0.0545\,\texttt{Price} +1.1996\,\texttt{US}$. (f) Both explain $23.9\%$ of the variation; dropping `Urban` lowers $R^2$ in the fifth decimal and raises $R_a^2$ from $0.2335$ to $0.2354$, and the partial $F$ for `Urban` is $0.007$, the square of its $t$ ratio. (g) `Price` $(-0.0648,-0.0442)$ and `US` $(0.692,1.708)$, neither containing $0$ (Proposition 3.1). (h) The largest standardised residual is $2.87$ in absolute value, and $23$ exceed $2$, against $18$ expected under normality: no outliers of note. Twenty observations exceed the leverage flag $2(k+1)/n=0.015$, the largest being $0.043$, six times the mean leverage: mild high leverage and no single dominating point. ■

**Exercise 3.9** (ISLR §3.7, exercise 15, part (d)). For the per-capita crime rate `crim` in `Boston`: is there evidence of a non-linear association with any predictor? For each predictor $X$ fit $Y=\beta_0+\beta_1X+\beta_2X^2+\beta_3X^3+\varepsilon$.

**Solution.** Non-linearity in $X$ is $H_0\colon\beta_2=\beta_3=0$, tested by the partial $F$ of Corollary 3.9 comparing the cubic with the linear fit. The indicator `chas` is excluded, since $\texttt{chas}^2=\texttt{chas}^3=\texttt{chas}$.

In [ ]:
preds <- setdiff(names(Boston), c("crim", "chas"))
round(t(sapply(preds, function(v) {
  m1 <- lm(reformulate(v, "crim"), data = Boston)
  m3 <- lm(reformulate(c(v, paste0("I(", v, "^2)"), paste0("I(", v, "^3)")), "crim"), data = Boston)
  a <- anova(m1, m3); c(F = a$F[2], p = a[2, "Pr(>F)"])
})), 5)

All eleven reject linearity at $5\%$; the evidence is strongest for `medv` ($F=117$), `dis`, `nox` and `indus`, and weakest for `lstat` ($p=0.037$) and `rad` ($p=0.026$). ■

**Exercise 3.10** (SOA sample question 13). Which of the following are true for a simple linear relation $y=\beta_0+\beta_1x+\varepsilon$? I. If $\varepsilon=0$, the $95\%$ confidence interval equals the $95\%$ prediction interval. II. The prediction interval is always at least as wide as the confidence interval. III. The prediction interval quantifies the possible range for $\operatorname{\mathsf{E}}(y\mid x)$. (A) I only; (B) II only; (C) III only; (D) I, II and III; (E) the correct answer is not given by (A), (B), (C) or (D).

**Solution.** By Unit 2's theorem **confidence and prediction intervals**, both are centred at $\hat y_0$, with half-widths $t\,s\sqrt{h_0}$ for $\operatorname{\mathsf{E}}(y\mid x_0)$ and $t\,s\sqrt{1+h_0}$ for a new $y$. I is true: with $\varepsilon=0$ every residual is $0$, so $s=0$ and both intervals are the single point $\hat y_0$. II is true: $\sqrt{1+h_0}>\sqrt{h_0}$, so the prediction interval contains the confidence interval. III is false: the range for $\operatorname{\mathsf{E}}(y\mid x)$ is the confidence interval; the prediction interval is for a new observation of $y$. I and II are true, a combination none of (A) to (D) offers, so the answer is (E). ■

**Exercise 3.11** (SOA sample question 27). Monthly dental claims over $48$ months are regressed on the number of weekdays, the number of weekend days and the average number of insured members:

|  | Coefficient | Standard error | $t$ | $p$-value |
|:--|--:|--:|--:|--:|
| Intercept | $-45{,}765{,}767.76$ | $20{,}441{,}816.55$ | $-2.24$ | $0.0303$ |
| Weekdays | $513{,}280.76$ | $233{,}143.23$ | $2.20$ | $0.0330$ |
| Weekend days | $280{,}148.46$ | $483{,}001.55$ | $0.58$ | $0.5649$ |
| Average members | $38.64$ | $6.42$ | $6.01$ | $0.0000$ |

Which predictors should be dropped at the $5\%$ level? (A) None; (B) weekdays only; (C) weekend days only; (D) average number of members; (E) all three.

**Solution.** Each $p$-value is that of the two-sided $t$ test of $H_0\colon\beta_j=0$ on $48-4=44$ degrees of freedom, whose $5\%$ critical value is $t_{44,0.975}=2.015$:

In [ ]:
qt(0.975, 44)

Weekend days have $t=0.58$, $p=0.56$: not significant, so the variable is dropped. Weekdays, $t=2.20$ with $p=0.033$, and members, $t=6.01$, are significant and kept. The answer is (C). (A) keeps an insignificant predictor; (B) and (D) drop significant ones; (E) drops all three. The intercept is not a predictor, and its $p$-value plays no role. Only one variable is dropped, so its $t$ test is equivalent to the partial $F$ test by Unit 2's proposition **$t^2=F$ for one explanatory variable**; dropping two would need the joint test of Corollary 3.9, since the remaining $t$ ratios change once a variable leaves the model. ■

**Exercise 3.12** (SOA sample question 49). A regression on $n$ observations gives a $95\%$ confidence interval $(t,u)$ for $y$ at given predictors and a $95\%$ prediction interval $(v,w)$ at the same predictors. Which must be true? I. $\lim_{n\to\infty}(u-t)=0$. II. $\lim_{n\to\infty}(w-v)=0$. III. $w-v>u-t$. (A) None; (B) I and II only; (C) I and III only; (D) II and III only; (E) the correct answer is not given by (A), (B), (C) or (D).

**Solution.** By Unit 2's theorem **confidence and prediction intervals**, $u-t=2c\,s\sqrt{h_0}$ and $w-v=2c\,s\sqrt{1+h_0}$ with $c=t_{n-k-1,0.975}$ and $h_0=\mathbf{x}_0'(\mathbf{X}'\mathbf{X})^{-1}\mathbf{x}_0$. By the prerequisites' proposition **quantiles of the $t$ law**, $c\geqslant z_{0.975}$ and $c\to z_{0.975}$. Under A5, $\mathrm{SSE}/\sigma^2\sim\chi^2_{n-k-1}$ (Unit 2), whose variance is $2(n-k-1)$ by the prerequisites' moments of $\chi^2$, so $\operatorname{var}(s^2)=2\sigma^4/(n-k-1)\to0$ and, by Chebyshev's inequality, $s^2\to\sigma^2$ in probability, hence $s\to\sigma$ in probability by the prerequisites' continuous mapping theorem. I holds whenever the design grows so that $h_0\to0$; for $k=1$, $h_0=1/n+(x_0-\bar x)^2/\{(n-1)s_x^2\}\to0$ when $s_x$ stays bounded away from $0$ and $\bar x$ stays bounded. II is false: $w-v\geqslant2z_{0.975}s$, which tends to $2z_{0.975}\sigma>0$. III is true whenever $s>0$, which holds with probability one under A5, since $\sqrt{1+h_0}>\sqrt{h_0}$. The answer is (C). ■

**Exercise 3.13** (SOA sample question 56). Which statement about prediction is true? (A) Each of several candidate regression models must produce the same prediction. (B) When making predictions, it is assumed that the new observation follows the same model as the one used in the sample. (C) A point prediction is more reliable than an interval prediction. (D) A wider prediction interval is more informative than a narrower one. (E) A prediction interval should not contain the single point prediction.

**Solution.** (B) is true: the prediction interval of Unit 2 is derived for a new response $Y_0=\mathbf{x}_0'\boldsymbol{\beta}+\varepsilon_0$ with the same $\boldsymbol{\beta}$ and $\sigma^2$ as the sample and $\varepsilon_0$ independent of it (§5.3). (A) is false: different models give different fitted functions, as the linear and quadratic fits of Example 3.5 do. (C) is false: the interval is built around the point prediction and quantifies its reliability, which the point alone does not report. (D) is false: at the same level, a narrower interval locates the new value more precisely. (E) is false: the interval $\hat y_0\pm c\,s\sqrt{1+h_0}$ is centred at the point prediction. ■

**Exercise 3.14** (SOA sample question 62). In a simple linear regression on over $100$ observations the estimated slope is $-1.03$ with standard error $0.06$. Calculate the $95\%$ confidence interval for the slope: (A) $(-1.15,-0.91)$; (B) $(-1.13,-0.93)$; (C) $(-1.11,-0.95)$; (D) $(-1.09,-0.97)$; (E) $(-1.07,-0.99)$.

**Solution.** By Unit 2's confidence interval, $-1.03\pm c\,(0.06)$ with $c=t_{n-2,0.975}\geqslant z_{0.975}$ by the prerequisites' proposition **quantiles of the $t$ law**:

In [ ]:
c(t_99 = qt(0.975, 99))
round(rbind(t_99 = -1.03 + c(-1, 1) * qt(0.975, 99) * 0.06,
            normal = -1.03 + c(-1, 1) * qnorm(0.975) * 0.06), 4)
round(c(B = 0.10, C = 0.08, D = 0.06, E = 0.04) / 0.06, 3)

Options (B) to (E) have half-widths $0.10$, $0.08$, $0.06$ and $0.04$, multipliers $1.67$, $1.33$, $1$ and $0.67$ of the standard error, all below $z_{0.975}\leqslant c$, so none is the interval, and (A) is the only option left; with the multipliers $z_{0.975}$ and $t_{99,0.975}=1.984$ both ends round to $(-1.15,-0.91)$, which is (A). ■

**Exercise 3.15** (SOA sample question 71). Which statement about the effect of omitting important variables from the regression model is NOT true? (A) It will result in underfitting. (B) It will increase the width of prediction intervals. (C) The coefficient estimates will remain unbiased. (D) The estimate of the variance, $s^2$, is inflated. (E) The total sum of squares remains unchanged.

**Solution.** (C) is not true, hence the answer: by Theorem 3.20(i) the coefficients of the included variables have bias $(\mathbf{X}_1'\mathbf{X}_1)^{-1}\mathbf{X}_1'\mathbf{X}_2\boldsymbol{\beta}_2$, which vanishes iff $\mathbf{X}_1'\mathbf{X}_2\boldsymbol{\beta}_2=\mathbf{0}$, for instance when the omitted variables are orthogonal to the included ones or have no effect, and is non-zero in general. (A) is the definition of underfitting (FREES §6.2.2). (D) is Theorem 3.20(iii): $\operatorname{\mathsf{E}}[s_1^2]$ exceeds $\sigma^2$ by $\sum_i\mathrm{Bias}(\hat Y_{1i})^2/(n-p)$. (B) follows from (D) as FREES states it: the prediction interval's width is proportional to $s$, and the inflated $s_1$ widens it, although in a given sample the change in leverage also enters. (E) is true because $\mathrm{TSS}=\sum_i(y_i-\bar y)^2$ does not involve the explanatory variables. ■